# Previsão do preço do Bitcoin com LSTM — o estudo em fases

**Objetivo:** descobrir o que **realmente** melhora a previsão quando mexemos no modelo, separando efeito de ruído.
O notebook segue a ordem das descobertas: **cada fase é uma tentativa de melhorar o modelo**, motivada pelo que a fase
anterior revelou. Dentro de cada fase, a metodologia é a mesma (ver `METODOLOGIA.md`): blocos sequenciais em que cada um
herda automaticamente o campeão do anterior, ruído entre seeds como régua, e o teste revelado só no fim da fase.

## Roteiro

- **Fase 1 — Ajustar os hiperparâmetros da lista (BTC 2017–2023)** — `lstm_b0_referencia`, `lstm_b1_entrada`, `lstm_b2_capacidade`, `lstm_b3_ativacao_init`, `lstm_b4_otimizacao`, `lstm_b4_checagem`, `lstm_b5_regularizacao`, `lstm_b6_complementar_capacidade`, `lstm_b7_complementar_epocas`, `lstm_b8_bonus_celula`, `lstm_b9_bonus_augmentation`, `lstm_b10_ablacao`
  - *Descoberta:* Na validação, a cadeia de blocos levou o Theil de 0,989 (LSTM padrão) para 0,979: ~2% melhor que o passeio aleatório. Os ganhos vieram da janela de 20 dias e da ativação ReLU na célula; capacidade, regularização, épocas e GRU não mudaram nada além do ruído.
- **Fase 2 — Revisão: configurações divergentes distorciam a análise?** — sem treino (reanálise)
  - *Descoberta:* Nenhuma decisão da fase 1 mudou: os 12 campeões são os mesmos, nenhuma configuração divergente chegou a uma final e a herança entre blocos é a mesma (divergências já ficavam no fim do ranking pelos erros enormes).
- **Fase 3 — Mais dados: série BTC-USD de 2014 a 2026** — `f3_referencia`, `f3_transplante`
  - *Descoberta:* Com mais dados, o LSTM padrão bateu o passeio aleatório no teste nas 5 seeds (Theil 0,983–0,995; seed 42: 0,988), ~1% melhor e acima do ruído. O ganho está no tamanho do movimento, não na direção: o acerto de subida/queda segue em ~50%.
- **Fase 4 — Informação de fora do preço e atenção** — `f4_contexto`, `f4_atencao`
  - *Descoberta:* Nenhuma fonte de contexto melhorou a validação: calendário, derivativos (funding) e sentimento empataram com só retornos dentro do ruído; on-chain e todas juntas pioraram além do ruído (Theil 0,997 e 0,996 contra 0,990). No teste, todas ficaram piores que só retornos (0,993 a 1,057 contra 0,988): mais informação levou a mais sobreajuste.
- **Fase 5 — O espaço de busca completo (série longa)** — `f5_referencia_modelos`, `f5_b1_entrada`, `f5_b2_normalizacao`, `f5_b3_arquitetura`, `f5_b4_ativacao_init`, `f5_b5_perda`, `f5_b6_otimizacao`, `f5_checagem`, `f5_b7_regularizacao`, `f5_b8_epocas`, `f5_bonus_augmentation`, `f5_ablacao`
  - *Descoberta:* Ainda não executada: esta fase é a próxima a rodar (os números aparecem nas células de análise).
- **Fase 6 — Lacunas da fase 5: ativações, dropout recorrente, horizonte e TimeGAN** — `f6_b1_ativacao_init`, `f6_b2_regularizacao`, `f6_horizonte`, `f6_bonus_timegan`
  - *Descoberta:* Ainda não executada.
- **Fase 7 — Mais dados reais: várias criptomoedas no treino** — `f7_referencia`, `f7_ativos`, `f7_bonus_timegan`
  - *Descoberta:* Ainda não executada.
- **Fase 8 — Mais dados reais: dados por hora** — `f8_referencia`, `f8_janela`
  - *Descoberta:* Ainda não executada.

As descobertas acima são das execuções de referência (fase 1 no Kaggle com 2× T4; fases 3 e 4 numa GPU local). Ao rodar,
o notebook recalcula tudo, e as células de análise de cada fase mostram os números desta execução.

## Protocolo (vale para todas as fases)

- **Validação walk-forward em 5 folds:** o fold k treina em tudo antes do bloco de validação k e valida nele. As partições são por data e são as mesmas em todos os experimentos da fase (comparações pareadas); um embargo de h dias impede que alvos do treino entrem na validação.
- **Métrica de decisão:** `val/rmse` (menor é melhor). Também são registradas `mse`, `mae`, `mape` (% no preço), `theil` (erro ÷ erro do passeio aleatório; < 1 = bate a referência), **`pocid`** (% de acerto na direção: D_t = 1 se (P_t − P_t−1)(P̂_t − P̂_t−1) > 0), `da` (acerto de direção em relação ao preço atual), `skill`, `r2` e `ic`.
- **Triagem e confirmação:** cada configuração roda os folds [3, 4, 5]; as 3 melhores completam os 5 folds, e o campeão é a melhor média nos 5 folds (nunca a média parcial da triagem).
- **Divergência:** um fold diverge se `val/theil` > 10 ou se a métrica não é finita. Configurações divergentes não chegam a finais, não viram campeãs e ficam fora das médias (listadas à parte).
- **Ruído entre seeds:** o LSTM padrão roda com 5 seeds na referência de cada série; um efeito só é "real" se passar de 2× o desvio da diferença pareada entre seeds.
- **Teste:** usado só no fim de cada fase, para os campeões e as referências.

**Resultados em disco** (W&B e GitHub são só espelhos): cada série tem sua pasta, `outputs/` (série 2017–2023) e
`outputs_btc_longo/` (série 2014–2026). Em cada uma: `{exp_name}/` por experimento (parâmetros, histórico por época,
resultados por fold, pesos, previsões), `_grids/{bloco}/` por bloco (configurações, rankings, campeão, logs) e
`_relatorio/` com tabelas e figuras.

**Tempo estimado:** a fase 1 tem ~1.000 treinos de fold (~1–3 h no Kaggle com 2× T4); as fases 3 e 4 somam ~100 treinos na
série longa (~20–40 min); a fase 5 tem ~1.200 treinos na série longa (~2–4 h); as fases 6 a 8 somam ~600 treinos, parte
deles com dezenas de milhares de janelas (várias moedas, dados por hora) e com o TimeGAN (~2–4 h). Para rodar só uma parte,
use `FASES` (e `RESUME_FROM` para trazer os resultados das fases que já rodaram). Se a sessão cair ou passar de 12 h, retome (veja abaixo): o que terminou é pulado.

## Como executar

O notebook é o mesmo em qualquer ambiente: clona o código do GitHub (ou usa a cópia local), acha os dados onde estiverem
e pula o que não estiver disponível. **Nada além de Python + GPU é obrigatório** (W&B e GitHub são opcionais).

### No Kaggle (execução principal)

1. **Dados:** os CSVs vêm no repositório clonado (série curta na raiz, série longa e séries externas em `data/`). Anexar o
   CSV curto como Dataset é opcional: o notebook o acha em `/kaggle/input` pelo nome ou pelas colunas `date`/`close`.
2. **Notebook:** *Code → New Notebook → File → Import Notebook* → `Kaggle_LSTM.ipynb`.
3. **Secrets:** *Add-ons → Secrets* → `GITHUB_TOKEN` e `WANDB_API_KEY`, marcados como anexados.
4. **Settings:** *Accelerator* **GPU T4 ×2**, *Internet* **On** (clone do código, pip, W&B e envio dos resultados).
5. *Save Version → **Save & Run All (Commit)***: roda em segundo plano (limite de 12 h); a saída fica em *Output*.
6. Para validar antes em minutos: `MODO_TESTE = True` (3 configurações por bloco, 2 épocas, pastas `*_teste`).

### No Colab ou Jupyter local

- **Colab:** *Runtime → Change runtime type → GPU*; o código é clonado; tokens em *Colab Secrets*; saída em `/content`.
- **Jupyter local:** abra o notebook dentro do repositório; tokens no `.env` (modelo em `.env.example`); saída no repositório.

**W&B.** Uma run por configuração, agrupada por bloco, num projeto por série (`if702-miniproject-2-lstm` e
`if702-miniproject-2-lstm-longo`); uma run de resumo por bloco (hiperparâmetro × métrica) e uma por fase. O teste só é
enviado na run de resumo da fase, depois de revelado.

**GitHub.** Com `GITHUB_TOKEN` com permissão de escrita (*Contents: Read and write*), ao fim de cada bloco os resultados vão
para o branch `resultados`, em `RUN_NAME/<pasta>/`.

**Retomada.** Para continuar uma execução interrompida: o mesmo `RUN_NAME` com `RESUME_FROM = "github"`, ou
`RESUME_FROM = "/kaggle/input/<output anterior>"` (a pasta que contém `outputs/` e `outputs_btc_longo/`). Para reaproveitar
uma fase já rodada (ex.: a fase 1 do Kaggle), basta que a pasta dela esteja lá: nada que já terminou é treinado de novo.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
import os

REPO_URL = "https://github.com/diegoflyra/if702-miniproject-2.git"  # repositório com src/, grids/, config/
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfão só de resultados
RUN_NAME = ""                    # vazio = "lstm-AAAAMMDD-HHMM"; fixe um nome para retomar pelo GitHub
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
FASES = ["fase1", "fase2", "fase3", "fase4", "fase5", "fase6", "fase7", "fase8"]  # fases a executar (as demais só são lidas, se já tiverem resultados)
WORKERS_PER_GPU = 2              # experimentos simultâneos por GPU (LSTMs pequenos; o Kaggle tem 4 CPUs)
CPU_WORKERS = 1                  # sem GPU
DATA_PATH = ""                   # CSV (ou pasta) da série curta; vazio = raiz do repo → /kaggle/input → /content
SYNC_PESOS = False               # enviar também os .pth ao GitHub
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "")
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida (3 configs por bloco, 2 épocas) em pastas *_teste, para validar o
                                 # notebook inteiro em minutos antes da execução completa

In [ ]:
import os
import shutil
import subprocess
import sys
import time


def _load_dotenv(path=".env"):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


_load_dotenv()


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(path):
    return all(os.path.exists(os.path.join(path, p)) for p in ("src/grid_search.py", "grids", "config/estudo.json"))


GITHUB_TOKEN = _secret("GITHUB_TOKEN")
here = os.getcwd()
if _is_repo(here):
    REPO_DIR = here
    print(f"Código: cópia local em {REPO_DIR}")
else:
    REPO_DIR = "/tmp/lstm-acoes"
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    if _is_repo(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    else:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        cloned = subprocess.run(["git", "clone", "-q", clone_url, REPO_DIR])
        if cloned.returncode != 0 or not _is_repo(REPO_DIR):
            raise RuntimeError("Falha ao clonar o repositório. Repositório privado exige GITHUB_TOKEN "
                               "(Kaggle Secrets, Colab Secrets ou variável de ambiente), ou abra o notebook "
                               "a partir de uma cópia local do projeto (pasta com src/, grids/ e config/).")
    print(f"Código: {REPO_DIR}")

os.chdir(REPO_DIR)
%cd {REPO_DIR}
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
import importlib.util

skip, pkgs = [], []
with open("requirements.txt", encoding="utf-8") as f:
    for line in f:
        pkg = line.strip()
        if not pkg or pkg.startswith("#"):
            continue
        name = pkg.split("==")[0].split(">=")[0].split("<=")[0].split("~=")[0].split("[")[0].strip().lower()
        if name == "torch" and importlib.util.find_spec(name) is not None:
            skip.append(name)
            continue
        pkgs.append(pkg)
if skip:
    print("Já instalado, não reinstalar (preserva CUDA do ambiente):", ", ".join(skip))
if pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import json

import pandas as pd

if os.path.isdir("/kaggle/working"):
    WORKDIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORKDIR = "/content"
else:
    WORKDIR = REPO_DIR
os.environ["EXP_OUTPUT_ROOT"] = WORKDIR
os.environ["EXP_OUTPUT_SUFFIX"] = "_teste" if MODO_TESTE else ""  # o teste nunca se mistura com a execução real
os.environ.pop("EXP_OUTPUT_DIR", None)
GRID_EXTRA = "--max_configs 3 --epochs 2" if MODO_TESTE else ""
GRID_EXTRA_B0 = "--epochs 2" if MODO_TESTE else ""  # a referência roda completa: as 5 seeds medem o ruído
if MODO_TESTE:
    print("MODO_TESTE: 3 configurações por bloco, 2 épocas, pastas *_teste (não servem como resultado).")

RUN_NAME = RUN_NAME or time.strftime("lstm-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
os.environ.update({"RUN_NAME": RUN_NAME, "RESULTS_REPO_URL": RESULTS_REPO_URL, "RESULTS_BRANCH": RESULTS_BRANCH,
                   "GITHUB_TOKEN": GITHUB_TOKEN, "WANDB_ENTITY": WANDB_ENTITY})
if DATA_PATH:
    os.environ["DATA_PATH"] = DATA_PATH
print(f"RUN_NAME = {RUN_NAME} | resultados em {WORKDIR}")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import common
import data as data_mod
import github_sync
import report_utils as rep
import wandb_report

if RESUME_FROM == "github":
    github_sync.pull(WORKDIR)
elif RESUME_FROM:
    subdirs = [d for d in os.listdir(RESUME_FROM) if d.startswith("outputs") and os.path.isdir(os.path.join(RESUME_FROM, d))]
    for d in subdirs or [os.path.basename(os.path.normpath(RESUME_FROM))]:
        src = os.path.join(RESUME_FROM, d) if subdirs else RESUME_FROM
        shutil.copytree(src, os.path.join(WORKDIR, d), dirs_exist_ok=True)
    print(f"Resultados anteriores copiados de {RESUME_FROM}; o que já foi concluído será pulado.")

wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ.pop("WANDB_MODE", None)
    print("W&B: chave carregada.")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print(f"W&B desativado ({'USAR_WANDB = False' if not USAR_WANDB else 'sem chave'}).")
if not (GITHUB_TOKEN and RESULTS_REPO_URL):
    print("GitHub: sem GITHUB_TOKEN/RESULTS_REPO_URL, os resultados ficam só em disco (e nos .zip).")


def usar_estudo(config):
    """Troca a série/estudo ativo: dados, partições, pasta de resultados e projeto do W&B."""
    global OUTPUTS
    os.environ["ESTUDO_CONFIG"] = config
    est = common.load_study()
    os.environ["WANDB_PROJECT"] = est.get("wandb_project", "if702-miniproject-2-lstm")
    OUTPUTS = common.output_dir()
    print(f"Estudo ativo: {est.get('titulo')} ({config}) → {OUTPUTS} | W&B: {os.environ['WANDB_PROJECT']}")


def executar(fase):
    """True se a fase está em FASES; senão as células de treino dela são puladas (e as análises leem o que houver)."""
    if fase not in FASES:
        print(f"{fase} fora de FASES: treino pulado; as análises abaixo usam os resultados já existentes, se houver.")
    return fase in FASES


def backup(bloco=""):
    """Backup parcial ao fim de cada bloco: <pasta>.zip (sem .pth) + espelho no GitHub, se configurado."""
    import zipfile

    zpath = OUTPUTS + ".zip"
    with zipfile.ZipFile(zpath + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(OUTPUTS):
            for name in files:
                if not name.endswith((".pth", ".tmp")):
                    full = os.path.join(root, name)
                    z.write(full, os.path.relpath(full, WORKDIR))
    os.replace(zpath + ".tmp", zpath)
    print(f"{os.path.basename(zpath)}: {os.path.getsize(zpath) / 1e6:.1f} MB")
    github_sync.push(OUTPUTS, weights=SYNC_PESOS, message=f"{RUN_NAME}: {bloco or 'backup'}")

In [ ]:
# Verificações rápidas antes de gastar GPU (sem treino): GPUs, modelos, métricas, augmentation e todos os grids
if shutil.which("nvidia-smi"):
    !nvidia-smi -L
else:
    print("nvidia-smi não encontrado; o treino usa o device que o PyTorch enxergar (CPU se não houver GPU).")
!python tests/check_models.py
!python tests/check_metrics.py
!python tests/check_augment.py
!python tests/check_grids.py

# Fase 1 — Ajustar os hiperparâmetros da lista (BTC 2017–2023)

**Motivação.** A pergunta inicial: quais hiperparâmetros do LSTM (camadas e nós, unidades densas, dropout, inicialização, decaimento, ativações, taxa de aprendizagem) realmente melhoram a previsão do Bitcoin? Blocos sequenciais, cada um herdando o campeão do anterior, com ruído entre seeds como régua, checagem de interação, bônus (GRU, data augmentation) e ablação do campeão.

**Blocos desta fase:**

- `lstm_b0_referencia` — Bloco 0 — Referência: Onde partimos? Quanto as previsões ingênuas e um LSTM com os valores iniciais recomendados acertam? (passeio_aleatorio, media_historica, ultimo_retorno, regressao_linear, lstm_padrao, lstm_padrao_seed1, lstm_padrao_seed2, lstm_padrao_seed3, lstm_padrao_seed4)
- `lstm_b1_entrada` — Bloco 1 — Representação da entrada: O que a rede deve ver: quantos dias de histórico, quais features e qual alvo? (lookback × features × target)
- `lstm_b2_capacidade` — Bloco 2 — Camadas ocultas, nós e camada densa: Quantas camadas LSTM ocultas, quantos nós por camada e quantas unidades densas absorvem o problema? (num_layers × hidden_size × fc_neurons (aleatória, 50))
- `lstm_b3_ativacao_init` — Bloco 3 — Funções de ativação e inicialização dos pesos: Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã? (lstm_activation × activation × weight_init (aleatória, 40))
- `lstm_b4_otimizacao` — Bloco 4 — Otimização: taxa de aprendizagem, decaimento, momentum e batch: Qual algoritmo, taxa de aprendizagem, taxa de decaimento, momentum e tamanho de batch fazem a rede convergir melhor? (optimizer × lr × decay_rate × momentum × batch_size (aleatória, 60))
- `lstm_b4_checagem` — Checagem de interação: Com a nova otimização, a estrutura campeã do Bloco 2 continua sendo a melhor? (2, 3º de `lstm_b2_capacidade` com a config. atual)
- `lstm_b5_regularizacao` — Bloco 5 — Dropout e decaimento dos pesos: Com a rede convergindo bem, quanto dropout e quanta penalização dos pesos controlam o overfitting ao ruído do mercado? (dropout × rnn_dropout × input_dropout × weight_decay (aleatória, 50))
- `lstm_b6_complementar_capacidade` — Complementar A — Camadas e nós sob regularização: Com o dropout ajustado, redes com mais nós ou mais camadas voltam a compensar? (num_layers × hidden_size)
- `lstm_b7_complementar_epocas` — Complementar B — Número de épocas (early stopping): O resultado está limitado pelo orçamento de épocas? Quanta paciência o early stopping deve ter? (patience)
- `lstm_b8_bonus_celula` — Bônus — LSTM × GRU: A célula LSTM é necessária, ou uma GRU (menos portas, menos parâmetros) chega ao mesmo resultado? (cell)
- `lstm_b9_bonus_augmentation` — Bônus — Data augmentation: Criar variações plausíveis das janelas de treino (ruído, amplitude, velocidade, ordem, recorte) melhora a previsão do campeão? (augment × aug_strength)
- `lstm_b10_ablacao` — Ablação — o que realmente importa no campeão: Das mudanças que levaram do LSTM padrão ao campeão, quais realmente melhoram o modelo e quais são dispensáveis? (desfaz cada mudança do campeão, uma por vez)

**Série:** `data-bitcoin_timedata-2023_v2 - data-bitcoin_timedata-2023_v2.csv`, de 2017-08-17 a 2023-08-01; 5 folds de validação de 180 dias; teste a partir de 2022-08-01.

In [ ]:
usar_estudo("config/estudo.json")
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())
!python tests/check_data.py
!python tests/check_features.py

## Bloco 0 — Referência

**Pergunta:** Onde partimos? Quanto as previsões ingênuas e um LSTM com os valores iniciais recomendados acertam?

Em séries financeiras, a referência difícil de bater é o **passeio aleatório** (prever que o preço de amanhã é o de hoje): `theil` < 1 e `skill` > 0 significam batê-lo, e a **POCID** mostra se o modelo acerta a direção da cotação. Também entram a média histórica, o último retorno (momentum ingênuo), uma **regressão linear** sobre a mesma janela (ser recorrente ajuda?) e o **LSTM padrão**, com os pontos de partida da lista: 1 camada oculta de 50 nós, densa de 10 unidades com ReLU, dropout de 20%, tanh na célula, Adam com lr 0,001 e decaimento 0,97 por época. Ele é a base do Bloco 1. **Ruído entre seeds:** o LSTM padrão é treinado com 5 seeds (42, 1, 2, 3, 4). A diferença entre duas execuções que só mudam a seed é a régua do estudo: uma mudança de hiperparâmetro só "melhora de verdade" se o ganho passar de ~2× esse ruído e se repetir na maioria dos folds.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Configuração | Parâmetros |
|---|---|
| `passeio_aleatorio` | `model=naive_zero` |
| `media_historica` | `model=naive_mean` |
| `ultimo_retorno` | `model=naive_last` |
| `regressao_linear` | `model=linear` |
| `lstm_padrao` | `model=lstm` |
| `lstm_padrao_seed1` | `model=lstm`, `seed=1` |
| `lstm_padrao_seed2` | `model=lstm`, `seed=2` |
| `lstm_padrao_seed3` | `model=lstm`, `seed=3` |
| `lstm_padrao_seed4` | `model=lstm`, `seed=4` |

**9 configurações.** Todas as configurações rodam os 5 folds.

**O que observar:** Se o LSTM padrão bate o passeio aleatório (`val/theil` < 1) e se a POCID passa de 50% de forma consistente entre folds. E o tamanho do ruído entre seeds, que define o menor efeito detectável.

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b0_referencia.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA_B0

In [ ]:
rep.discarded_configs("lstm_b0_referencia")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b0_referencia")  # treinaram, mas divergiram (val/theil > limite): fora das médias

In [ ]:
rep.plot_grid_bars("lstm_b0_referencia")

#### Confirmação das finalistas e campeão — `lstm_b0_referencia`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b0_referencia", "final")

In [ ]:
rep.show_champion("lstm_b0_referencia")
rep.plot_finalists("lstm_b0_referencia")

#### Ruído entre seeds

A régua do estudo: o menor efeito que se distingue de sorte na inicialização.

In [ ]:
ruido = rep.noise_floor()
rep.noise_floor("val/pocid")

### 📝 Análise — Bloco 0

- **Algum modelo bateu o passeio aleatório (theil < 1, skill > 0)?** _…_
- **O LSTM padrão superou a regressão linear na mesma janela?** _…_
- **POCID e acurácia direcional: acima de 50% de forma consistente?** _…_
- **Ruído entre seeds: qual o menor efeito que o estudo consegue detectar?** _…_

In [ ]:
backup("lstm_b0_referencia")

## Bloco 1 — Representação da entrada

**Pergunta:** O que a rede deve ver: quantos dias de histórico, quais features e qual alvo?

Não está na lista de hiperparâmetros da rede, mas vem antes dela: a janela e as features definem o problema que o LSTM resolve. O alvo `close` (preço normalizado) é comparado com `log_return`; as métricas são sempre calculadas da mesma forma, então os dois são comparáveis.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Eixo | Valores |
|---|---|
| `lookback` | `5`, `10`, `20`, `40`, `60`, `120` |
| `features` | `retornos`, `retornos_volume`, `ohlcv`, `tecnicos` |
| `target` | `log_return`, `close` |

**48 combinações.** Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** Se janelas longas ajudam ou só trazem ruído, se volume, amplitude e indicadores acrescentam algo sobre só retornos, e se o alvo em preço extrapola mal fora da faixa do treino.

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b1_entrada.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `lstm_b1_entrada`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b1_entrada", "triagem")

In [ ]:
rep.discarded_configs("lstm_b1_entrada")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b1_entrada")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b1_entrada`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b1_entrada")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b1_entrada", row="lookback", col="features", facet="target")

In [ ]:
rep.heatmap("lstm_b1_entrada", row="lookback", col="features", facet="target", value="val/pocid_mean")

In [ ]:
rep.param_effect("lstm_b1_entrada")

In [ ]:
rep.plot_grid_bars("lstm_b1_entrada")

#### Confirmação das finalistas e campeão — `lstm_b1_entrada`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b1_entrada", "final")

In [ ]:
rep.show_champion("lstm_b1_entrada")
rep.plot_finalists("lstm_b1_entrada")

### 📝 Análise — Bloco 1

- **Janela: qual tamanho foi útil?** _…_
- **Features: o que acrescentou sobre só retornos?** _…_
- **Alvo log_return × close:** _…_
- **Diferença do campeão para o LSTM padrão (validação):** _…_

In [ ]:
backup("lstm_b1_entrada")

## Bloco 2 — Camadas ocultas, nós e camada densa

**Pergunta:** Quantas camadas LSTM ocultas, quantos nós por camada e quantas unidades densas absorvem o problema?

Itens 1 e 2 da lista. Regra prática: 1 camada oculta basta para problemas simples e 2 para os razoavelmente complexos; muitos nós aumentam a capacidade (com regularização) e poucos causam subajuste. Na camada densa, 5–10 unidades são um bom ponto de partida (`[]` = sem densa; `[10, 10]` = duas densas). Espaço de 3 × 7 × 7 = 147 combinações, com **busca aleatória** de 50; o dropout de 20% do padrão é mantido.

**Base:** o melhor campeão entre `lstm_b1_entrada`.

| Eixo | Valores |
|---|---|
| `num_layers` | `1`, `2`, `3` |
| `hidden_size` | `8`, `16`, `32`, `50`, `64`, `128`, `256` |
| `fc_neurons` | `[]`, `[5]`, `[10]`, `[25]`, `[50]`, `[10, 10]`, `[25, 10]` |

**147 combinações possíveis; busca aleatória de 50** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds. Limite: 2,000,000 parâmetros.

**O que observar:** Onde a capacidade satura, se mais camadas só aumentam o gap treino–validação, e se a densa ajuda ou atrapalha.

In [ ]:
rep.show_champion("lstm_b1_entrada")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b2_capacidade.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `lstm_b2_capacidade`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b2_capacidade", "triagem")

In [ ]:
rep.discarded_configs("lstm_b2_capacidade")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b2_capacidade")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b2_capacidade`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b2_capacidade")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b2_capacidade", row="hidden_size", col="num_layers")

In [ ]:
rep.heatmap("lstm_b2_capacidade", row="hidden_size", col="num_layers", value="gap/rmse_mean")

In [ ]:
rep.heatmap("lstm_b2_capacidade", row="fc_neurons", col="num_layers")

In [ ]:
rep.param_effect("lstm_b2_capacidade")

In [ ]:
rep.plot_grid_bars("lstm_b2_capacidade")

#### Confirmação das finalistas e campeão — `lstm_b2_capacidade`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b2_capacidade", "final")

In [ ]:
rep.show_champion("lstm_b2_capacidade")
rep.plot_finalists("lstm_b2_capacidade")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b2_capacidade"), "lstm_b2_capacidade__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b2_capacidade"), "lstm_b2_capacidade__*", metric="val/pocid")

### 📝 Análise — Bloco 2

- **Camadas ocultas: 1, 2 ou 3?** _…_
- **Nós por camada: onde satura? Houve subajuste com poucos nós?** _…_
- **Camada densa (unidades):** _…_
- **Ganho sobre o Bloco 1 (validação):** _…_

In [ ]:
backup("lstm_b2_capacidade")

## Bloco 3 — Funções de ativação e inicialização dos pesos

**Pergunta:** Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã?

Itens 4 e 6 da lista. **Célula LSTM:** as portas são sempre sigmoid; varia a ativação da candidata e da saída (tanh é o padrão; sigmoid, softsign e ReLU usam uma célula própria, mais lenta que o cuDNN). **Densas:** ReLU, tanh, sigmoid e ELU; se a campeã não tiver camada densa, esse eixo não tem efeito e as combinações equivalentes são descartadas. A saída é sempre linear, porque o alvo é contínuo (sigmoid e softmax na saída são para classificação). **Inicialização:** pesos pequenos e aleatórios; `padrao` = PyTorch U(±1/√h), `uniforme` = U(±0,05), `normal` = N(0; 0,05), `xavier` (Glorot), `glorot_ortogonal` (padrão do Keras: ortogonal na recorrência e bias de esquecimento 1) e `he` (Kaiming, pensado para ReLU). Espaço de 4 × 4 × 6 = 96 combinações, com busca aleatória de 40.

**Base:** o melhor campeão entre `lstm_b2_capacidade`.

| Eixo | Valores |
|---|---|
| `lstm_activation` | `tanh`, `sigmoid`, `softsign`, `relu` |
| `activation` | `relu`, `tanh`, `sigmoid`, `elu` |
| `weight_init` | `padrao`, `uniforme`, `normal`, `xavier`, `glorot_ortogonal`, `he` |

**96 combinações possíveis; busca aleatória de 40** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** Se a ativação da célula muda algo além de tanh (ReLU pode explodir sem limite), e se a inicialização afeta a convergência (`melhor_epoca_media`) e a variância entre folds.

In [ ]:
rep.show_champion("lstm_b2_capacidade")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b3_ativacao_init.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `lstm_b3_ativacao_init`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b3_ativacao_init", "triagem")

In [ ]:
rep.discarded_configs("lstm_b3_ativacao_init")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b3_ativacao_init")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b3_ativacao_init`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b3_ativacao_init")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b3_ativacao_init", row="weight_init", col="lstm_activation")

In [ ]:
rep.heatmap("lstm_b3_ativacao_init", row="weight_init", col="lstm_activation", value="melhor_epoca_media")

In [ ]:
rep.heatmap("lstm_b3_ativacao_init", row="activation", col="lstm_activation")

In [ ]:
rep.param_effect("lstm_b3_ativacao_init")

In [ ]:
rep.plot_grid_bars("lstm_b3_ativacao_init")

#### Confirmação das finalistas e campeão — `lstm_b3_ativacao_init`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b3_ativacao_init", "final")

In [ ]:
rep.show_champion("lstm_b3_ativacao_init")
rep.plot_finalists("lstm_b3_ativacao_init")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b3_ativacao_init"), "lstm_b3_ativacao_init__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b3_ativacao_init"), "lstm_b3_ativacao_init__*", metric="val/pocid")

### 📝 Análise — Bloco 3

- **Ativação da célula LSTM:** _…_
- **Ativação das densas:** _…_
- **Inicialização: afetou a convergência ou só o ruído entre folds?** _…_
- **Ganho sobre o Bloco 2 (validação):** _…_

In [ ]:
backup("lstm_b3_ativacao_init")

## Bloco 4 — Otimização: taxa de aprendizagem, decaimento, momentum e batch

**Pergunta:** Qual algoritmo, taxa de aprendizagem, taxa de decaimento, momentum e tamanho de batch fazem a rede convergir melhor?

Item 7 (taxa de aprendizagem, entre 0 e 0,1, de preferência com decaimento) e item 5 (taxa de decaimento, com ponto de partida 0,97). Aqui o **decaimento é aplicado à taxa de aprendizagem**: lr ← lr × `decay_rate` a cada época, e 1,0 = sem decaimento. O decaimento dos **pesos** (penalização L2, `weight_decay`) é regularização e fica no Bloco 5. Também entram momentum (SGD e RMSprop; o Adam não usa) e tamanho de batch. O grid de lr é comum a todos os algoritmos de propósito, para mostrar a faixa útil de cada um. Espaço de 3 × 7 × 5 × 4 × 5 = 2.100 combinações, com **busca aleatória** de 60; combinações equivalentes (momentum no Adam) são descartadas sem gastar o orçamento.

**Base:** o melhor campeão entre `lstm_b3_ativacao_init`.

| Eixo | Valores |
|---|---|
| `optimizer` | `sgd`, `adam`, `rmsprop` |
| `lr` | `0.0001`, `0.0003`, `0.001`, `0.003`, `0.01`, `0.03`, `0.1` |
| `decay_rate` | `1`, `0.99`, `0.97`, `0.95`, `0.9` |
| `momentum` | `0`, `0.5`, `0.9`, `0.99` |
| `batch_size` | `16`, `32`, `64`, `128`, `256` |

**2100 combinações possíveis; busca aleatória de 60** (seed 0). Fixos no bloco: `scheduler=exponential`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** A faixa útil de lr de cada otimizador, onde ele diverge (lr 0,1), se o decaimento estabiliza lr altas, e se batch pequeno (gradiente mais ruidoso) regulariza.

In [ ]:
rep.show_champion("lstm_b3_ativacao_init")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b4_otimizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `lstm_b4_otimizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b4_otimizacao", "triagem")

In [ ]:
rep.discarded_configs("lstm_b4_otimizacao")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b4_otimizacao")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b4_otimizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b4_otimizacao")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b4_otimizacao", row="optimizer", col="lr")

In [ ]:
rep.heatmap("lstm_b4_otimizacao", row="decay_rate", col="lr")

In [ ]:
rep.heatmap("lstm_b4_otimizacao", row="optimizer", col="lr", value="melhor_epoca_media")

In [ ]:
rep.param_effect("lstm_b4_otimizacao")

In [ ]:
rep.plot_grid_bars("lstm_b4_otimizacao")

#### Confirmação das finalistas e campeão — `lstm_b4_otimizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b4_otimizacao", "final")

In [ ]:
rep.show_champion("lstm_b4_otimizacao")
rep.plot_finalists("lstm_b4_otimizacao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b4_otimizacao"), "lstm_b4_otimizacao__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b4_otimizacao"), "lstm_b4_otimizacao__*", metric="val/pocid")

### 📝 Análise — Bloco 4

- **Faixa de lr útil de cada otimizador:** _…_
- **Houve divergência?** _…_
- **Decaimento da lr: 0,97 é um bom valor? Interage com a lr?** _…_
- **Momentum e batch size:** _…_
- **Ganho sobre o Bloco 3 (validação):** _…_

In [ ]:
backup("lstm_b4_otimizacao")

## Checagem de interação

**Pergunta:** Com a nova otimização, a estrutura campeã do Bloco 2 continua sendo a melhor?

O 2º e o 3º colocados do Bloco 2 (camadas, nós e densa) são treinados com a ativação, a inicialização e a otimização campeãs (K folds). O Bloco 5 parte da melhor rede entre o campeão do Bloco 4 e esta checagem.

**Base:** o melhor campeão entre `lstm_b4_otimizacao`.

Todas as configurações rodam os 5 folds.

**O que observar:** Se a ordem das estruturas se inverte com a nova otimização.

In [ ]:
rep.show_champion("lstm_b4_otimizacao")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b4_checagem.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("lstm_b4_checagem")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b4_checagem")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### Confirmação das finalistas e campeão — `lstm_b4_checagem`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b4_checagem", "final")

In [ ]:
rep.show_champion("lstm_b4_checagem")
rep.plot_finalists("lstm_b4_checagem")

In [ ]:
pd.concat([rep.grid_ranking("lstm_b4_otimizacao", "final").head(1),
           rep.grid_ranking("lstm_b4_checagem", "final")], ignore_index=True)

### 📝 Análise — Checagem de interação

- **A ordem das estruturas se manteve com a nova otimização?** _…_

In [ ]:
backup("lstm_b4_checagem")

## Bloco 5 — Dropout e decaimento dos pesos

**Pergunta:** Com a rede convergindo bem, quanto dropout e quanta penalização dos pesos controlam o overfitting ao ruído do mercado?

Item 3 da lista, mais o decaimento dos pesos do item 5. Todo LSTM é acompanhado de dropout: `rnn_dropout` entre camadas LSTM empilhadas (só vale com 2+ camadas; com 1 camada é equivalente a 0 e é descartado), `dropout` depois da última LSTM e entre as densas (nunca na saída), e `input_dropout` na entrada. O recomendado é partir de 20% e não passar de 50%. `weight_decay` é a penalização L2, que encolhe os pesos a cada atualização. Espaço de 6 × 6 × 3 × 5 = 540 combinações, com busca aleatória de 50. Mais épocas e paciência, porque a regularização atrasa a convergência.

**Base:** o melhor campeão entre `lstm_b4_otimizacao`, `lstm_b4_checagem`.

| Eixo | Valores |
|---|---|
| `dropout` | `0`, `0.1`, `0.2`, `0.3`, `0.4`, `0.5` |
| `rnn_dropout` | `0`, `0.1`, `0.2`, `0.3`, `0.4`, `0.5` |
| `input_dropout` | `0`, `0.1`, `0.2` |
| `weight_decay` | `0`, `1e-06`, `1e-05`, `0.0001`, `0.001` |

**540 combinações possíveis; busca aleatória de 50** (seed 0). Fixos no bloco: `epochs=200`, `patience=20`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O gap treino–validação (`gap/rmse`) e a melhor época: o dropout atrasa a decoreba? Houve subajuste com 50%?

In [ ]:
rep.show_champion("lstm_b4_otimizacao")
rep.show_champion("lstm_b4_checagem")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b5_regularizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `lstm_b5_regularizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b5_regularizacao", "triagem")

In [ ]:
rep.discarded_configs("lstm_b5_regularizacao")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b5_regularizacao")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b5_regularizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b5_regularizacao")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b5_regularizacao", row="dropout", col="weight_decay")

In [ ]:
rep.heatmap("lstm_b5_regularizacao", row="dropout", col="weight_decay", value="gap/rmse_mean")

In [ ]:
rep.heatmap("lstm_b5_regularizacao", row="dropout", col="rnn_dropout")

In [ ]:
rep.param_effect("lstm_b5_regularizacao")

In [ ]:
rep.plot_grid_bars("lstm_b5_regularizacao")

#### Confirmação das finalistas e campeão — `lstm_b5_regularizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b5_regularizacao", "final")

In [ ]:
rep.show_champion("lstm_b5_regularizacao")
rep.plot_finalists("lstm_b5_regularizacao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b5_regularizacao"), "lstm_b5_regularizacao__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b5_regularizacao"), "lstm_b5_regularizacao__*", metric="val/pocid")

### 📝 Análise — Bloco 5

- **Dropout: 20% foi o melhor compromisso?** _…_
- **Dropout entre camadas e na entrada:** _…_
- **Decaimento dos pesos (L2):** _…_
- **Gap treino–validação e subajuste:** _…_
- **Ganho sobre o Bloco 4 (validação):** _…_

In [ ]:
backup("lstm_b5_regularizacao")

## Complementar A — Camadas e nós sob regularização

**Pergunta:** Com o dropout ajustado, redes com mais nós ou mais camadas voltam a compensar?

Checagem de interação entre decisões distantes no tempo: o Bloco 2 escolheu camadas e nós só com o dropout inicial. A lista diz que muitos nós *com regularização* podem aumentar a precisão, e este bloco testa isso. A combinação idêntica ao campeão do Bloco 5 é re-treinada e serve de **checagem de reprodutibilidade**.

**Base:** o melhor campeão entre `lstm_b5_regularizacao`.

| Eixo | Valores |
|---|---|
| `num_layers` | `1`, `2`, `3` |
| `hidden_size` | `32`, `64`, `128`, `256` |

**12 combinações.** A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds. Limite: 2,000,000 parâmetros.

**O que observar:** O heatmap camadas × nós, o gap, e se a repetição reproduz a validação do Bloco 5.

In [ ]:
rep.show_champion("lstm_b5_regularizacao")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b6_complementar_capacidade.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `lstm_b6_complementar_capacidade`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b6_complementar_capacidade", "triagem")

In [ ]:
rep.discarded_configs("lstm_b6_complementar_capacidade")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b6_complementar_capacidade")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b6_complementar_capacidade`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b6_complementar_capacidade")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b6_complementar_capacidade", row="hidden_size", col="num_layers")

In [ ]:
rep.heatmap("lstm_b6_complementar_capacidade", row="hidden_size", col="num_layers", value="gap/rmse_mean")

In [ ]:
rep.plot_grid_bars("lstm_b6_complementar_capacidade")

#### Confirmação das finalistas e campeão — `lstm_b6_complementar_capacidade`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b6_complementar_capacidade", "final")

In [ ]:
rep.show_champion("lstm_b6_complementar_capacidade")
rep.plot_finalists("lstm_b6_complementar_capacidade")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b6_complementar_capacidade"), "lstm_b6_complementar_capacidade__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b6_complementar_capacidade"), "lstm_b6_complementar_capacidade__*", metric="val/pocid")

### 📝 Análise — Complementar A

- **Com regularização, a capacidade ideal mudou?** _…_
- **A repetição reproduziu o campeão do Bloco 5? Qual o ruído entre execuções?** _…_

In [ ]:
backup("lstm_b6_complementar_capacidade")

## Complementar B — Número de épocas (early stopping)

**Pergunta:** O resultado está limitado pelo orçamento de épocas? Quanta paciência o early stopping deve ter?

O número de épocas é decidido pelo early stopping na `val/loss`, e os pesos da melhor época são restaurados. O limite sobe para 500 épocas e varia só a paciência. Com decaimento da lr, paciência longa também dá tempo para a lr cair. O campeão deste bloco é o **resultado principal** do estudo.

**Base:** o melhor campeão entre `lstm_b6_complementar_capacidade`.

| Eixo | Valores |
|---|---|
| `patience` | `5`, `10`, `20`, `40` |

**4 combinações.** Fixos no bloco: `epochs=500`. A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Se a `melhor_epoca_media` encosta no limite e se paciência maior melhora a validação ou só gasta GPU.

In [ ]:
rep.show_champion("lstm_b6_complementar_capacidade")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b7_complementar_epocas.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("lstm_b7_complementar_epocas")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b7_complementar_epocas")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b7_complementar_epocas`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b7_complementar_epocas")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("lstm_b7_complementar_epocas")

#### Confirmação das finalistas e campeão — `lstm_b7_complementar_epocas`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b7_complementar_epocas", "final")

In [ ]:
rep.show_champion("lstm_b7_complementar_epocas")
rep.plot_finalists("lstm_b7_complementar_epocas")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b7_complementar_epocas"), "lstm_b7_complementar_epocas__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b7_complementar_epocas"), "lstm_b7_complementar_epocas__*", metric="val/pocid")

### 📝 Análise — Complementar B

- **Paciência maior ajudou? O resultado estava limitado pelas épocas?** _…_
- **Custo (tempo) × ganho:** _…_

In [ ]:
backup("lstm_b7_complementar_epocas")

## Bônus — LSTM × GRU

**Pergunta:** A célula LSTM é necessária, ou uma GRU (menos portas, menos parâmetros) chega ao mesmo resultado?

Fora da sequência principal: troca só a célula recorrente do campeão principal. A GRU usa a ativação tanh padrão. A receita campeã (LSTM) é re-treinada neste bloco como referência pareada.

**Base:** o melhor campeão entre `lstm_b7_complementar_epocas`.

| Eixo | Valores |
|---|---|
| `cell` | `lstm`, `gru` |

**2 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** A diferença pareada fold a fold, o número de parâmetros e o tempo de treino.

In [ ]:
rep.show_champion("lstm_b7_complementar_epocas")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b8_bonus_celula.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("lstm_b8_bonus_celula")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b8_bonus_celula")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b8_bonus_celula`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b8_bonus_celula")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("lstm_b8_bonus_celula")

#### Confirmação das finalistas e campeão — `lstm_b8_bonus_celula`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b8_bonus_celula", "final")

In [ ]:
rep.show_champion("lstm_b8_bonus_celula")
rep.plot_finalists("lstm_b8_bonus_celula")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b8_bonus_celula"), "lstm_b8_bonus_celula__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b8_bonus_celula"), "lstm_b8_bonus_celula__*", metric="val/pocid")

### 📝 Análise — Bônus

- **GRU × LSTM (pareado):** _…_
- **Diferença de parâmetros e de tempo de treino:** _…_

In [ ]:
backup("lstm_b8_bonus_celula")

## Bônus — Data augmentation

**Pergunta:** Criar variações plausíveis das janelas de treino (ruído, amplitude, velocidade, ordem, recorte) melhora a previsão do campeão?

Data augmentation é **pré-processamento**, não hiperparâmetro da rede, por isso fica fora da sequência principal (como no estudo da CNN). A cada época, cada janela de treino é transformada com probabilidade 50%; validação e teste usam as janelas originais. As janelas estão normalizadas (z-score), então as intensidades são em desvios padrão. **Jittering:** ruído gaussiano (σ 0,03 fraca / 0,1 forte). **Scaling:** multiplica a janela por um fator ~ N(1, σ) (σ 0,1 / 0,2), simulando regimes de volatilidade; com alvo em retorno, o alvo é escalado junto. **Magnitude warping:** multiplica por uma curva suave aleatória (σ 0,1 / 0,2). **Time warping:** acelera e desacelera trechos da janela (σ 0,1 / 0,2), preservando o dia da decisão. **Permutation:** embaralha 3 ou 6 segmentos da janela; se não piorar, o modelo não está usando a ordem temporal. **Window slicing:** recorta 90% ou 70% da janela e reamostra para o tamanho original (a janela deslizante com sobreposição já é como as amostras são montadas). Também entra a combinação clássica jitter + scaling. **Base:** o campeão principal, re-treinado aqui sem augmentation como referência pareada, com mais épocas e paciência para todos, porque augmentation retarda a convergência. Em séries financeiras o ganho não é garantido; o bloco mede quais estratégias ajudam e quais atrapalham.

**Base:** o melhor campeão entre `lstm_b7_complementar_epocas`.

| Eixo | Valores |
|---|---|
| `augment` | `none`, `jitter`, `scaling`, `magwarp`, `timewarp`, `permutation`, `slicing`, `jitter+scaling` |
| `aug_strength` | `fraca`, `forte` |

**16 combinações.** Fixos no bloco: `epochs=300`, `patience=30`, `aug_prob=0.5`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O ganho pareado sobre a referência sem augmentation, a queda do gap treino–validação, e se a permutação piora (o que confirma que o LSTM usa a ordem temporal).

In [ ]:
rep.show_champion("lstm_b7_complementar_epocas")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b9_bonus_augmentation.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `lstm_b9_bonus_augmentation`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b9_bonus_augmentation", "triagem")

In [ ]:
rep.discarded_configs("lstm_b9_bonus_augmentation")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("lstm_b9_bonus_augmentation")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `lstm_b9_bonus_augmentation`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b9_bonus_augmentation")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b9_bonus_augmentation", row="augment", col="aug_strength")

In [ ]:
rep.heatmap("lstm_b9_bonus_augmentation", row="augment", col="aug_strength", value="gap/rmse_mean")

In [ ]:
rep.heatmap("lstm_b9_bonus_augmentation", row="augment", col="aug_strength", value="val/pocid_mean")

In [ ]:
rep.plot_grid_bars("lstm_b9_bonus_augmentation")

#### Confirmação das finalistas e campeão — `lstm_b9_bonus_augmentation`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b9_bonus_augmentation", "final")

In [ ]:
rep.show_champion("lstm_b9_bonus_augmentation")
rep.plot_finalists("lstm_b9_bonus_augmentation")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b9_bonus_augmentation"), "lstm_b9_bonus_augmentation__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b9_bonus_augmentation"), "lstm_b9_bonus_augmentation__*", metric="val/pocid")

### 📝 Análise — Bônus

- **Alguma estratégia melhorou a validação além do ruído entre seeds?** _…_
- **Qual estratégia reduziu mais o gap treino–validação?** _…_
- **Permutação: o modelo depende da ordem temporal?** _…_
- **Time warping e slicing: distorcer o tempo ajuda ou atrapalha em retornos diários?** _…_
- **Intensidade fraca × forte:** _…_

In [ ]:
backup("lstm_b9_bonus_augmentation")

## Ablação — o que realmente importa no campeão

**Pergunta:** Das mudanças que levaram do LSTM padrão ao campeão, quais realmente melhoram o modelo e quais são dispensáveis?

Cada configuração parte do campeão principal e **desfaz uma única mudança**, voltando aquele hiperparâmetro ao valor do padrão do estudo. Todas rodam os K folds e são comparadas fold a fold com o campeão re-treinado aqui. Se piora ao desfazer (além do ruído entre seeds), a mudança ajuda; se fica dentro do ruído, é dispensável; se melhora ao desfazer, a mudança atrapalhava e só entrou por sorte na seleção. Hiperparâmetros cuja reversão não muda nada (ex.: momentum com Adam) são descartados como equivalentes. É a resposta mais direta para "o que melhora o modelo quando mexemos".

**Base:** o melhor campeão entre `lstm_b7_complementar_epocas`.

A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Quais mudanças ficam verdes (ajudam além do ruído) e se as mais importantes batem com a importância dos eixos em cada bloco.

In [ ]:
rep.show_champion("lstm_b7_complementar_epocas")

In [ ]:
if executar("fase1"):
    !python src/grid_search.py grids/lstm_b10_ablacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultado da ablação — `lstm_b10_ablacao`

Δ = (campeão sem a mudança) − (campeão), fold a fold. Verde: a mudança ajuda além do ruído entre seeds; cinza: dispensável; vermelho: atrapalhava.

In [ ]:
rep.discarded_configs("lstm_b10_ablacao")  # mudanças cuja reversão não altera nada

In [ ]:
rep.ablation_table("lstm_b10_ablacao")

### 📝 Análise — Ablação

- **Quais mudanças realmente ajudam (além do ruído)?** _…_
- **Quais são dispensáveis e poderiam voltar ao padrão?** _…_
- **Alguma mudança atrapalhava?** _…_
- **Isso confirma a importância dos eixos vista em cada bloco?** _…_

In [ ]:
backup("lstm_b10_ablacao")

## Fase 1 · O que realmente melhorou o modelo (validação)

A **cadeia de decisões** reconstrói, pela herança entre blocos, o caminho do LSTM padrão até o campeão (`lstm_b7_complementar_epocas`): o que
mudou em cada passo, o Δ pareado por fold, em quantos folds a mudança venceu e se o ganho passa do ruído entre seeds.

In [ ]:
cadeia = rep.decision_chain("lstm_b7_complementar_epocas")
cadeia

## Fase 1 · Teste revelado

Até aqui todas as escolhas desta fase usaram só a validação. Agora o período de teste é usado **uma única vez**, para as
referências e o campeão de cada bloco (média ± desvio entre os modelos dos K folds). A concordância entre validação e
teste é a evidência de que não houve vazamento; uma discrepância grande (ex.: mudança de regime) é um achado.

In [ ]:
final = rep.final_report(["lstm_b0_referencia", "lstm_b1_entrada", "lstm_b2_capacidade", "lstm_b3_ativacao_init", "lstm_b4_otimizacao", "lstm_b4_checagem", "lstm_b5_regularizacao", "lstm_b6_complementar_capacidade", "lstm_b7_complementar_epocas", "lstm_b8_bonus_celula", "lstm_b9_bonus_augmentation"])
final

In [ ]:
campeao_fase = rep.champion_name("lstm_b7_complementar_epocas")
rep.plot_predictions(campeao_fase)

### O que descobrimos nesta fase

Na execução de referência:

- Na validação, a cadeia de blocos levou o Theil de 0,989 (LSTM padrão) para 0,979: ~2% melhor que o passeio aleatório. Os ganhos vieram da janela de 20 dias e da ativação ReLU na célula; capacidade, regularização, épocas e GRU não mudaram nada além do ruído.
- No teste (2022-08 → 2023-08), nenhum LSTM bateu o passeio aleatório: o campeão ficou com Theil 1,019, pior até que o LSTM padrão (1,003). Acerto de direção ~50%. Os ganhos da validação eram, em boa parte, o efeito de escolher o melhor entre ~300 configurações, e o teste caiu num regime bem mais calmo.
- Só retornos como entrada foi melhor que volume, OHLCV ou indicadores técnicos; o alvo em preço quebrou (extrapolação na alta de 2021). Data augmentation piorou 13 das 14 variações.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Fase 2 — Revisão: configurações divergentes distorciam a análise?

**Motivação.** 48 das 309 configurações da fase 1 tiveram fold divergente: todas as de alvo em preço (folds de 2021) e as que combinavam ReLU na célula com RMSprop e lr alta (redes maiores, a checagem de interação inteira, parte da ablação). A seleção de campeões e as médias dos gráficos ainda as consideravam. Correção: um fold diverge se val/theil > 10 ou se a métrica não é finita; configurações divergentes não chegam a finais nem viram campeãs, e ficam fora das médias.

**Série:** `data-bitcoin_timedata-2023_v2 - data-bitcoin_timedata-2023_v2.csv`, de 2017-08-17 a 2023-08-01; 5 folds de validação de 180 dias; teste a partir de 2022-08-01.

In [ ]:
usar_estudo("config/estudo.json")

## Quantas configurações divergiram, e o que elas têm em comum

In [ ]:
por_bloco, padroes = rep.divergence_summary(["lstm_b0_referencia", "lstm_b1_entrada", "lstm_b2_capacidade", "lstm_b3_ativacao_init", "lstm_b4_otimizacao", "lstm_b4_checagem", "lstm_b5_regularizacao", "lstm_b6_complementar_capacidade", "lstm_b7_complementar_epocas", "lstm_b8_bonus_celula", "lstm_b9_bonus_augmentation"])
display(por_bloco)
display(padroes)

## A correção mudou alguma decisão?

Para cada bloco: finalistas divergentes, se o campeão divergiu e se ele é o melhor entre os não divergentes.

In [ ]:
rep.selection_check(["lstm_b0_referencia", "lstm_b1_entrada", "lstm_b2_capacidade", "lstm_b3_ativacao_init", "lstm_b4_otimizacao", "lstm_b4_checagem", "lstm_b5_regularizacao", "lstm_b6_complementar_capacidade", "lstm_b7_complementar_epocas", "lstm_b8_bonus_celula", "lstm_b9_bonus_augmentation"])

### O que descobrimos nesta fase

Na execução de referência:

- Nenhuma decisão da fase 1 mudou: os 12 campeões são os mesmos, nenhuma configuração divergente chegou a uma final e a herança entre blocos é a mesma (divergências já ficavam no fim do ranking pelos erros enormes).
- Mudaram só as médias dos gráficos de efeito, que misturavam configurações explodidas. A conclusão da fase 1 se mantém.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Fase 3 — Mais dados: série BTC-USD de 2014 a 2026

**Motivação.** A série da fase 1 tem só 6 anos e o teste caiu num período atípico. Com o dobro de história (BTC-USD do Yahoo, 2014-09 → 2026-09), cada fold treina em mais regimes de mercado e o teste passa a ser o último ano (2025-09 → 2026-09). Duas perguntas: o LSTM padrão passa a bater o passeio aleatório? O campeão da fase 1 continua bom aqui?

**Blocos desta fase:**

- `f3_referencia` — Fase 3 · Referência na série longa: Com o dobro de história (2014–2026) e um teste recente, o LSTM padrão passa a bater o passeio aleatório? (passeio_aleatorio, media_historica, ultimo_retorno, regressao_linear, lstm_padrao, lstm_padrao_seed1, lstm_padrao_seed2, lstm_padrao_seed3, lstm_padrao_seed4)
- `f3_transplante` — Fase 3 · Transplante do campeão da fase 1: Os hiperparâmetros ajustados na fase 1 continuam bons numa série maior, ou eram sobreajuste à série curta? (campeao_fase1, campeao_fase1_tanh)

**Série:** `data/btc-usd_yahoo_2014-09-17_2026-09-23.csv`, de 2014-09-17 a 2026-09-23; 5 folds de validação de 365 dias; teste a partir de 2025-09-24.

**Ressalva:** A escolha de uma série mais longa foi motivada pelo teste da fase 1. Para não reaproveitar aquele teste, esta fase usa outro período de teste (2025-09 → 2026-09).

In [ ]:
usar_estudo("config/estudo_btc_longo.json")
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())
!python tests/check_data.py
!python tests/check_features.py

## Fase 3 · Referência na série longa

**Pergunta:** Com o dobro de história (2014–2026) e um teste recente, o LSTM padrão passa a bater o passeio aleatório?

Mesmo protocolo e mesmas referências do Bloco 0 da fase 1, agora na série BTC-USD de 2014-09 a 2026-09 (Yahoo Finance): cada bloco de validação tem 1 ano, e o treino do fold 1 já cobre 6 anos. O LSTM padrão roda com 5 seeds, que dão a régua de ruído desta série.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Configuração | Parâmetros |
|---|---|
| `passeio_aleatorio` | `model=naive_zero` |
| `media_historica` | `model=naive_mean` |
| `ultimo_retorno` | `model=naive_last` |
| `regressao_linear` | `model=linear` |
| `lstm_padrao` | `model=lstm` |
| `lstm_padrao_seed1` | `model=lstm`, `seed=1` |
| `lstm_padrao_seed2` | `model=lstm`, `seed=2` |
| `lstm_padrao_seed3` | `model=lstm`, `seed=3` |
| `lstm_padrao_seed4` | `model=lstm`, `seed=4` |

**9 configurações.** Todas as configurações rodam os 5 folds.

**O que observar:** Se o Theil de teste do LSTM padrão fica abaixo de 1 em todas as seeds, e se a acurácia de direção sai de ~50%.

In [ ]:
if executar("fase3"):
    !python src/grid_search.py grids/f3_referencia.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA_B0

In [ ]:
rep.discarded_configs("f3_referencia")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f3_referencia")  # treinaram, mas divergiram (val/theil > limite): fora das médias

In [ ]:
rep.plot_grid_bars("f3_referencia")

#### Confirmação das finalistas e campeão — `f3_referencia`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f3_referencia", "final")

In [ ]:
rep.show_champion("f3_referencia")
rep.plot_finalists("f3_referencia")

#### Ruído entre seeds

A régua do estudo: o menor efeito que se distingue de sorte na inicialização.

In [ ]:
ruido = rep.noise_floor()
rep.noise_floor("val/pocid")

### 📝 Análise — Fase 3 · Referência na série longa

- **Algum modelo bateu o passeio aleatório (theil < 1, skill > 0)?** _…_
- **O LSTM padrão superou a regressão linear na mesma janela?** _…_
- **POCID e acurácia direcional: acima de 50% de forma consistente?** _…_
- **Ruído entre seeds: qual o menor efeito que o estudo consegue detectar?** _…_

In [ ]:
backup("f3_referencia")

## Fase 3 · Transplante do campeão da fase 1

**Pergunta:** Os hiperparâmetros ajustados na fase 1 continuam bons numa série maior, ou eram sobreajuste à série curta?

O campeão principal da fase 1 é treinado sem mudanças na série longa, e também com a célula LSTM em tanh (a ativação ReLU na célula se mostrou instável na fase 1). A comparação pareada é contra o LSTM padrão da referência desta fase.

**Configurações:** `campeao_fase1` = campeão de `lstm_b7_complementar_epocas` (config/estudo.json); `campeao_fase1_tanh` = campeão de `lstm_b7_complementar_epocas` (config/estudo.json) com {'lstm_activation': 'tanh'}.

Todas as configurações rodam os 5 folds.

**O que observar:** Se o campeão da fase 1 bate o LSTM padrão na série longa, ou se fica pior (sinal de que o ajuste fino era específico da série curta).

In [ ]:
if executar("fase3"):
    !python src/grid_search.py grids/f3_transplante.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f3_transplante")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f3_transplante")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### Confirmação das finalistas e campeão — `f3_transplante`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f3_transplante", "final")

In [ ]:
rep.show_champion("f3_transplante")
rep.plot_finalists("f3_transplante")

#### Comparação pareada com `f3_referencia__lstm_padrao` (validação)

In [ ]:
rep.paired_comparison("f3_referencia__lstm_padrao", "f3_transplante__*")

### 📝 Análise — Fase 3 · Transplante do campeão da fase 1

- **O campeão da fase 1 bateu o LSTM padrão na série longa?** _…_
- **Trocar a célula para tanh mudou algo?** _…_
- **O que isso diz sobre o ajuste fino da fase 1?** _…_

In [ ]:
backup("f3_transplante")

## Fase 3 · Teste revelado

Até aqui todas as escolhas desta fase usaram só a validação. Agora o período de teste é usado **uma única vez**, para as
referências e o campeão de cada bloco (média ± desvio entre os modelos dos K folds). A concordância entre validação e
teste é a evidência de que não houve vazamento; uma discrepância grande (ex.: mudança de regime) é um achado.

In [ ]:
final = rep.final_report(["f3_referencia", "f3_transplante"])
final

In [ ]:
campeao_fase = rep.champion_name("f3_referencia")
rep.plot_predictions(campeao_fase)

### O que descobrimos nesta fase

Na execução de referência:

- Com mais dados, o LSTM padrão bateu o passeio aleatório no teste nas 5 seeds (Theil 0,983–0,995; seed 42: 0,988), ~1% melhor e acima do ruído. O ganho está no tamanho do movimento, não na direção: o acerto de subida/queda segue em ~50%.
- O campeão da fase 1 transplantado ficou pior que o LSTM padrão na validação (Theil 0,998 contra 0,990) e não bateu o passeio aleatório no teste (1,002; 1,009 com a célula em tanh): o ajuste fino da fase 1 era específico da série curta.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Fase 4 — Informação de fora do preço e atenção

**Motivação.** Se o retorno passado sozinho carrega pouca informação, talvez o que falte sejam dados de fora da série: derivativos (funding rate), fundamentos de rede (on-chain), sentimento (medo e ganância) e calendário (fim de semana). Também uma camada de atenção, que deixa a rede focar em dias específicos da janela. Parte do LSTM padrão, o melhor da fase 3.

**Blocos desta fase:**

- `f4_contexto` — Fase 4 · Contexto externo ao preço: Informação de fora da série de preços (calendário, derivativos, rede, sentimento) ajuda a prever o retorno? (features)
- `f4_atencao` — Fase 4 · Atenção após a LSTM: Uma camada de atenção, que aprende a focar em dias específicos da janela, supera o último estado da LSTM? (pooling)

**Série:** `data/btc-usd_yahoo_2014-09-17_2026-09-23.csv`, de 2014-09-17 a 2026-09-23; 5 folds de validação de 365 dias; teste a partir de 2025-09-24.

**Ressalva:** Esta fase usa o mesmo teste da fase 3; o teste já foi visto uma vez, então uma melhora aqui precisaria ser grande para ser convincente.

In [ ]:
usar_estudo("config/estudo_btc_longo.json")
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())
!python tests/check_data.py
!python tests/check_features.py

## Fase 4 · Contexto externo ao preço

**Pergunta:** Informação de fora da série de preços (calendário, derivativos, rede, sentimento) ajuda a prever o retorno?

Partindo do LSTM padrão (o melhor da fase 3), cada conjunto acrescenta ao retorno diário uma fonte de contexto. **Calendário:** seno e cosseno do dia da semana (o BTC negocia 7 dias por semana, e o fim de semana tem menos liquidez). **Derivativos:** taxa de financiamento do perpétuo da BitMEX (média do dia), seu z-score de 30 dias (extremos precedem liquidações) e um indicador de disponibilidade (o dado começa em 2016-05). **On-chain:** variação diária do número de transações e de endereços ativos, z-score de 30 dias do volume transacionado em US$ e variação de 7 dias do hash rate (blockchain.com). **Sentimento:** índice de medo e ganância, sua variação diária e um indicador de disponibilidade (começa em 2018-02). **Externos:** todos juntos. Antes do início de cada fonte, o valor é 0 e o indicador de disponibilidade vale 0. Open interest e long/short ratio ficaram de fora porque as APIs gratuitas só guardam 30 dias; netflows de exchanges e transações de baleias só existem em serviços pagos. Os dados estão congelados em `data/externos/`.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Eixo | Valores |
|---|---|
| `features` | `retornos`, `calendario`, `derivativos`, `onchain`, `sentimento`, `externos` |

**6 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Se alguma fonte bate o retorno sozinho além do ruído entre seeds, e se juntar tudo ajuda ou só aumenta o sobreajuste.

In [ ]:
if executar("fase4"):
    !python src/grid_search.py grids/f4_contexto.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f4_contexto")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f4_contexto")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f4_contexto`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f4_contexto")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f4_contexto")

#### Confirmação das finalistas e campeão — `f4_contexto`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f4_contexto", "final")

In [ ]:
rep.show_champion("f4_contexto")
rep.plot_finalists("f4_contexto")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f4_contexto"), "f4_contexto__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f4_contexto"), "f4_contexto__*", metric="val/pocid")

### 📝 Análise — Fase 4 · Contexto externo ao preço

- **Alguma fonte de contexto melhorou a validação além do ruído?** _…_
- **Derivativos (funding) × on-chain × sentimento: qual trouxe mais informação?** _…_
- **Juntar tudo ajudou ou atrapalhou?** _…_
- **O efeito aparece também na POCID (direção)?** _…_

In [ ]:
backup("f4_contexto")

## Fase 4 · Atenção após a LSTM

**Pergunta:** Uma camada de atenção, que aprende a focar em dias específicos da janela, supera o último estado da LSTM?

Sobre o campeão do bloco de contexto, varia só o resumo da sequência: `last` (estado do último dia, o padrão), `mean` (média dos estados no tempo) e `attention` (média ponderada com pesos aprendidos, que permite focar, por exemplo, num pico de volume há 5 dias).

**Base:** o melhor campeão entre `f4_contexto`.

| Eixo | Valores |
|---|---|
| `pooling` | `last`, `mean`, `attention` |

**3 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** O ganho pareado da atenção sobre o último estado, comparado com o ruído entre seeds.

In [ ]:
rep.show_champion("f4_contexto")

In [ ]:
if executar("fase4"):
    !python src/grid_search.py grids/f4_atencao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f4_atencao")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f4_atencao")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f4_atencao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f4_atencao")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f4_atencao")

#### Confirmação das finalistas e campeão — `f4_atencao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f4_atencao", "final")

In [ ]:
rep.show_champion("f4_atencao")
rep.plot_finalists("f4_atencao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f4_atencao"), "f4_atencao__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f4_atencao"), "f4_atencao__*", metric="val/pocid")

### 📝 Análise — Fase 4 · Atenção após a LSTM

- **A atenção melhorou a validação além do ruído?** _…_
- **E o teste?** _…_

In [ ]:
backup("f4_atencao")

## Fase 4 · O que realmente melhorou o modelo (validação)

A **cadeia de decisões** reconstrói, pela herança entre blocos, o caminho do LSTM padrão até o campeão (`f4_atencao`): o que
mudou em cada passo, o Δ pareado por fold, em quantos folds a mudança venceu e se o ganho passa do ruído entre seeds.

In [ ]:
cadeia = rep.decision_chain("f4_atencao")
cadeia

## Fase 4 · Teste revelado

Até aqui todas as escolhas desta fase usaram só a validação. Agora o período de teste é usado **uma única vez**, para as
referências e o campeão de cada bloco (média ± desvio entre os modelos dos K folds). A concordância entre validação e
teste é a evidência de que não houve vazamento; uma discrepância grande (ex.: mudança de regime) é um achado.

In [ ]:
final = rep.final_report(["f4_contexto", "f4_atencao"])
final

In [ ]:
campeao_fase = rep.champion_name("f4_atencao")
rep.plot_predictions(campeao_fase)

### O que descobrimos nesta fase

Na execução de referência:

- Nenhuma fonte de contexto melhorou a validação: calendário, derivativos (funding) e sentimento empataram com só retornos dentro do ruído; on-chain e todas juntas pioraram além do ruído (Theil 0,997 e 0,996 contra 0,990). No teste, todas ficaram piores que só retornos (0,993 a 1,057 contra 0,988): mais informação levou a mais sobreajuste.
- A atenção piorou: Theil de validação 1,002 contra 0,990 do último estado, pior em 5 de 5 folds (a média no tempo também piorou, 1,000). O campeão da fase continua sendo o LSTM padrão da fase 3.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Fase 5 — O espaço de busca completo (série longa)

**Motivação.** As fases anteriores exploraram um eixo de cada vez. Esta fase refaz a busca em blocos na série longa com tudo o que foi proposto ao longo do estudo e mais os eixos clássicos que tinham ficado de fora: outras famílias de modelos como referência (SVM, Random Forest, XGBoost, CNN, as referências do paper de Wu et al., 2025); todas as features (incluindo as de mercado do paper); normalizações (scaler, alvo pela volatilidade, por janela) e período de treino; arquitetura (LSTM/GRU, bidirecional, pilhas decrescentes, atenção, LayerNorm, residual, CNN-LSTM); ativações e inicialização; funções de erro (MSE, MAE, Huber, log-cosh, direcional); otimização com agendas da taxa; todas as formas de dropout, incluindo o recorrente; épocas; data augmentation; ablação; e fusão de modelos inspirada na Combinatorial Fusion Analysis do paper, sem o vazamento dele (pesos e escolha só pela validação).

**Blocos desta fase:**

- `f5_referencia_modelos` — Fase 5 · Outras famílias de modelos (referências do paper): Com as mesmas janelas, como o LSTM se compara a SVM, Random Forest, XGBoost, CNN e regressão linear? (svr, random_forest, xgboost, cnn1d, regressao_linear, lstm_padrao, lstm_paper, lstm_paper_preco)
- `f5_b1_entrada` — Fase 5 · Bloco 1 — Entrada: janela, features e alvo: Quantos dias de histórico, quais informações e qual alvo? (lookback × features × target (aleatória, 50))
- `f5_b2_normalizacao` — Fase 5 · Bloco 2 — Normalização e período de treino: Como normalizar entradas e alvo, e com quanto histórico treinar? (scaler × alvo_vol × norm_janela × treino_inicio)
- `f5_b3_arquitetura` — Fase 5 · Bloco 3 — Arquitetura: Que arquitetura recorrente funciona melhor? (cell × bidirectional × pilha × pooling × layer_norm × residual × conv_layers × fc_neurons (aleatória, 60))
- `f5_b4_ativacao_init` — Fase 5 · Bloco 4 — Ativações e inicialização: Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã? (lstm_activation × activation × weight_init (aleatória, 30))
- `f5_b5_perda` — Fase 5 · Bloco 5 — Função de erro do treino: Com qual função de erro a rede aprende melhor? (perda)
- `f5_b6_otimizacao` — Fase 5 · Bloco 6 — Otimização: Qual algoritmo, taxa de aprendizagem, agenda da taxa, momentum e batch? (optimizer × lr × agenda × momentum × batch_size (aleatória, 60))
- `f5_checagem` — Fase 5 · Checagem de interação: Com a otimização nova, a arquitetura campeã continua a melhor? (2, 3º de `f5_b3_arquitetura` com a config. atual)
- `f5_b7_regularizacao` — Fase 5 · Bloco 7 — Regularização: todos os dropouts: Quanto e onde aplicar dropout, e quanta penalização e corte de gradiente? (dropout × rnn_dropout × input_dropout × recurrent_dropout × weight_decay × grad_clip (aleatória, 60))
- `f5_b8_epocas` — Fase 5 · Bloco 8 — Número de épocas: O resultado está limitado pelo orçamento de épocas? Quanta paciência o early stopping deve ter? (patience)
- `f5_bonus_augmentation` — Fase 5 · Bônus — Data augmentation: Criar variações plausíveis das janelas de treino (ruído, amplitude, velocidade, ordem, recorte) melhora a previsão do campeão? (augment × aug_strength)
- `f5_ablacao` — Fase 5 · Ablação do campeão: Das mudanças que levaram do LSTM padrão ao campeão, quais realmente melhoram o modelo e quais são dispensáveis? (desfaz cada mudança do campeão, uma por vez)

**Série:** `data/btc-usd_yahoo_2014-09-17_2026-09-23.csv`, de 2014-09-17 a 2026-09-23; 5 folds de validação de 365 dias; teste a partir de 2025-09-24.

**Ressalva:** Mesmo teste das fases 3 e 4, já visto: uma melhora aqui precisa ser clara (além do ruído e consistente entre folds) para ser convincente.

In [ ]:
usar_estudo("config/estudo_btc_longo.json")
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())
!python tests/check_data.py
!python tests/check_features.py

## Fase 5 · Outras famílias de modelos (referências do paper)

**Pergunta:** Com as mesmas janelas, como o LSTM se compara a SVM, Random Forest, XGBoost, CNN e regressão linear?

Os cinco modelos base do paper de Wu et al. (IEEE CAI 2025), treinados nas mesmas janelas e com o mesmo alvo do LSTM padrão: SVR (kernel RBF), Random Forest (300 árvores), XGBoost (300 árvores), CNN 1D (duas convoluções) e regressão linear. Também o LSTM do paper (camadas de 100 e 50 unidades, dropout de 20%, normalização min-max, features de mercado + hash rate + EMA/MACD, treino a partir de 2020-03), com alvo em retorno e em preço, como eles fazem.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Configuração | Parâmetros |
|---|---|
| `svr` | `model=svr` |
| `random_forest` | `model=random_forest` |
| `xgboost` | `model=xgboost` |
| `cnn1d` | `model=cnn1d`, `conv_filters=32` |
| `regressao_linear` | `model=linear` |
| `lstm_padrao` | `model=lstm` |
| `lstm_paper` | `model=lstm`, `hidden_sizes=[100, 50]`, `dropout=0.2`, `scaler=minmax`, `features=paper`, `treino_inicio=2020-03-11` |
| `lstm_paper_preco` | `model=lstm`, `hidden_sizes=[100, 50]`, `dropout=0.2`, `scaler=minmax`, `features=paper`, `treino_inicio=2020-03-11`, `target=close` |

**8 configurações.** Todas as configurações rodam os 5 folds.

**O que observar:** Se alguma outra família bate o LSTM padrão, e o que acontece com a receita do paper fora do período em que foi publicada.

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_referencia_modelos.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f5_referencia_modelos")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_referencia_modelos")  # treinaram, mas divergiram (val/theil > limite): fora das médias

In [ ]:
rep.plot_grid_bars("f5_referencia_modelos")

#### Confirmação das finalistas e campeão — `f5_referencia_modelos`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_referencia_modelos", "final")

In [ ]:
rep.show_champion("f5_referencia_modelos")
rep.plot_finalists("f5_referencia_modelos")

### 📝 Análise — Fase 5 · Outras famílias de modelos (referências do paper)

- **Alguma família de modelos bateu o LSTM padrão?** _…_
- **A receita do paper funciona na nossa série e no nosso protocolo?** _…_
- **O alvo em preço (como no paper) se comporta como na fase 1?** _…_

In [ ]:
backup("f5_referencia_modelos")

## Fase 5 · Bloco 1 — Entrada: janela, features e alvo

**Pergunta:** Quantos dias de histórico, quais informações e qual alvo?

Todos os conjuntos de features do estudo: retornos, OHLCV, indicadores técnicos, EMA/MACD, calendário, derivativos (funding), on-chain, sentimento, mercado (ETH, ouro, S&P 500, VIX, juro, dólar, Nvidia, Tesla), a receita do paper e tudo junto. Espaço de 5 × 11 × 2 = 110 combinações, com busca aleatória de 50.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Eixo | Valores |
|---|---|
| `lookback` | `10`, `20`, `40`, `60`, `120` |
| `features` | `retornos`, `ohlcv`, `tecnicos`, `tecnicos_ema`, `calendario`, `derivativos`, `onchain`, `sentimento`, `mercado`, `paper`, `tudo` |
| `target` | `log_return`, `close` |

**110 combinações possíveis; busca aleatória de 50** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O efeito de cada conjunto de features controlando janela e alvo.

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_b1_entrada.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f5_b1_entrada`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f5_b1_entrada", "triagem")

In [ ]:
rep.discarded_configs("f5_b1_entrada")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_b1_entrada")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_b1_entrada`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_b1_entrada")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f5_b1_entrada", row="features", col="lookback")

In [ ]:
rep.heatmap("f5_b1_entrada", row="features", col="lookback", value="val/pocid_mean")

In [ ]:
rep.param_effect("f5_b1_entrada")

In [ ]:
rep.plot_grid_bars("f5_b1_entrada")

#### Confirmação das finalistas e campeão — `f5_b1_entrada`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_b1_entrada", "final")

In [ ]:
rep.show_champion("f5_b1_entrada")
rep.plot_finalists("f5_b1_entrada")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b1_entrada"), "f5_b1_entrada__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b1_entrada"), "f5_b1_entrada__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bloco 1

- **Janela:** _…_
- **Quais features trouxeram informação?** _…_
- **Alvo retorno × preço:** _…_

In [ ]:
backup("f5_b1_entrada")

## Fase 5 · Bloco 2 — Normalização e período de treino

**Pergunta:** Como normalizar entradas e alvo, e com quanto histórico treinar?

**Scaler** das features e do alvo (ajustado só no treino de cada fold): z-score, min-max [0, 1] (como no paper), robusto (mediana e IQR) ou nenhum. **Alvo pela volatilidade:** prever o retorno dividido pela volatilidade dos últimos 20 dias e multiplicar de volta (retornos em dias calmos e agitados ficam na mesma escala). **Normalização por janela:** cada janela vira o z-score dela mesma. **Início do treino:** todo o histórico, a partir de 2018 ou a partir de 2020-03 (pós-COVID, como no paper). Grid de 4 × 2 × 2 × 3 = 48.

**Base:** o melhor campeão entre `f5_b1_entrada`.

| Eixo | Valores |
|---|---|
| `scaler` | `standard`, `minmax`, `robust`, `none` |
| `alvo_vol` | `False`, `True` |
| `norm_janela` | `False`, `True` |
| `treino_inicio` | `None`, `2018-01-01`, `2020-03-11` |

**48 combinações.** A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** Se a normalização muda o resultado além do ruído, e se descartar o histórico antigo ajuda (regimes recentes) ou atrapalha (menos dados).

In [ ]:
rep.show_champion("f5_b1_entrada")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_b2_normalizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f5_b2_normalizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f5_b2_normalizacao", "triagem")

In [ ]:
rep.discarded_configs("f5_b2_normalizacao")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_b2_normalizacao")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_b2_normalizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_b2_normalizacao")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f5_b2_normalizacao", row="scaler", col="treino_inicio")

In [ ]:
rep.heatmap("f5_b2_normalizacao", row="alvo_vol", col="norm_janela")

In [ ]:
rep.param_effect("f5_b2_normalizacao")

In [ ]:
rep.plot_grid_bars("f5_b2_normalizacao")

#### Confirmação das finalistas e campeão — `f5_b2_normalizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_b2_normalizacao", "final")

In [ ]:
rep.show_champion("f5_b2_normalizacao")
rep.plot_finalists("f5_b2_normalizacao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b2_normalizacao"), "f5_b2_normalizacao__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b2_normalizacao"), "f5_b2_normalizacao__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bloco 2

- **Scaler:** _…_
- **Alvo normalizado pela volatilidade:** _…_
- **Normalização por janela:** _…_
- **Início do treino (mais dados × dados mais recentes):** _…_

In [ ]:
backup("f5_b2_normalizacao")

## Fase 5 · Bloco 3 — Arquitetura

**Pergunta:** Que arquitetura recorrente funciona melhor?

Célula LSTM ou GRU; unidirecional ou bidirecional; pilha de camadas (incluindo tamanhos decrescentes, como o 100 → 50 do paper); resumo da sequência (último estado, média, atenção); LayerNorm; conexões residuais; convoluções 1D antes da recorrência (CNN-LSTM) e camadas densas. Espaço de 2 × 2 × 6 × 3 × 2 × 2 × 3 × 3 = 2.592 combinações, com busca aleatória de 60.

**Base:** o melhor campeão entre `f5_b2_normalizacao`.

| Eixo | Valores |
|---|---|
| `cell` | `lstm`, `gru` |
| `bidirectional` | `False`, `True` |
| `pilha` | **1x50** (`hidden_sizes=[]`, `num_layers=1`, `hidden_size=50`), **1x100** (`hidden_sizes=[]`, `num_layers=1`, `hidden_size=100`), **2x64** (`hidden_sizes=[]`, `num_layers=2`, `hidden_size=64`), **3x64** (`hidden_sizes=[]`, `num_layers=3`, `hidden_size=64`), **100-50** (`hidden_sizes=[100, 50]`), **128-64-32** (`hidden_sizes=[128, 64, 32]`) |
| `pooling` | `last`, `mean`, `attention` |
| `layer_norm` | `False`, `True` |
| `residual` | `False`, `True` |
| `conv_layers` | `0`, `1`, `2` |
| `fc_neurons` | `[]`, `[10]`, `[25, 10]` |

**2592 combinações possíveis; busca aleatória de 60** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds. Limite: 2,000,000 parâmetros.

**O que observar:** Quais eixos explicam a variação (importância), e se alguma arquitetura bate a pilha simples além do ruído.

In [ ]:
rep.show_champion("f5_b2_normalizacao")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_b3_arquitetura.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f5_b3_arquitetura`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f5_b3_arquitetura", "triagem")

In [ ]:
rep.discarded_configs("f5_b3_arquitetura")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_b3_arquitetura")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_b3_arquitetura`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_b3_arquitetura")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f5_b3_arquitetura", row="pilha", col="cell")

In [ ]:
rep.heatmap("f5_b3_arquitetura", row="pooling", col="conv_layers")

In [ ]:
rep.param_effect("f5_b3_arquitetura")

In [ ]:
rep.plot_grid_bars("f5_b3_arquitetura")

#### Confirmação das finalistas e campeão — `f5_b3_arquitetura`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_b3_arquitetura", "final")

In [ ]:
rep.show_champion("f5_b3_arquitetura")
rep.plot_finalists("f5_b3_arquitetura")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b3_arquitetura"), "f5_b3_arquitetura__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b3_arquitetura"), "f5_b3_arquitetura__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bloco 3

- **LSTM × GRU:** _…_
- **Bidirecional:** _…_
- **Profundidade e pilhas decrescentes:** _…_
- **Atenção:** _…_
- **Residual e LayerNorm:** _…_
- **CNN-LSTM:** _…_

In [ ]:
backup("f5_b3_arquitetura")

## Fase 5 · Bloco 4 — Ativações e inicialização

**Pergunta:** Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã?

Itens 4 e 6 da lista. **Célula LSTM:** as portas são sempre sigmoid; varia a ativação da candidata e da saída (tanh é o padrão; sigmoid, softsign e ReLU usam uma célula própria, mais lenta que o cuDNN). **Densas:** ReLU, tanh, sigmoid e ELU; se a campeã não tiver camada densa, esse eixo não tem efeito e as combinações equivalentes são descartadas. A saída é sempre linear, porque o alvo é contínuo (sigmoid e softmax na saída são para classificação). **Inicialização:** pesos pequenos e aleatórios; `padrao` = PyTorch U(±1/√h), `uniforme` = U(±0,05), `normal` = N(0; 0,05), `xavier` (Glorot), `glorot_ortogonal` (padrão do Keras: ortogonal na recorrência e bias de esquecimento 1) e `he` (Kaiming, pensado para ReLU). Espaço de 4 × 4 × 6 = 96 combinações, com busca aleatória de 40.

**Base:** o melhor campeão entre `f5_b3_arquitetura`.

| Eixo | Valores |
|---|---|
| `lstm_activation` | `tanh`, `sigmoid`, `softsign`, `relu` |
| `activation` | `relu`, `tanh`, `sigmoid`, `elu` |
| `weight_init` | `padrao`, `uniforme`, `normal`, `xavier`, `glorot_ortogonal`, `he` |

**96 combinações possíveis; busca aleatória de 30** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** Se a ativação da célula muda algo além de tanh (ReLU pode explodir sem limite), e se a inicialização afeta a convergência (`melhor_epoca_media`) e a variância entre folds.

In [ ]:
rep.show_champion("f5_b3_arquitetura")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_b4_ativacao_init.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f5_b4_ativacao_init`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f5_b4_ativacao_init", "triagem")

In [ ]:
rep.discarded_configs("f5_b4_ativacao_init")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_b4_ativacao_init")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_b4_ativacao_init`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_b4_ativacao_init")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f5_b4_ativacao_init", row="weight_init", col="lstm_activation")

In [ ]:
rep.heatmap("f5_b4_ativacao_init", row="weight_init", col="lstm_activation", value="melhor_epoca_media")

In [ ]:
rep.heatmap("f5_b4_ativacao_init", row="activation", col="lstm_activation")

In [ ]:
rep.param_effect("f5_b4_ativacao_init")

In [ ]:
rep.plot_grid_bars("f5_b4_ativacao_init")

#### Confirmação das finalistas e campeão — `f5_b4_ativacao_init`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_b4_ativacao_init", "final")

In [ ]:
rep.show_champion("f5_b4_ativacao_init")
rep.plot_finalists("f5_b4_ativacao_init")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b4_ativacao_init"), "f5_b4_ativacao_init__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b4_ativacao_init"), "f5_b4_ativacao_init__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bloco 4

- **Ativação da célula LSTM:** _…_
- **Ativação das densas:** _…_
- **Inicialização: afetou a convergência ou só o ruído entre folds?** _…_
- **Ganho sobre o Bloco 2 (validação):** _…_

In [ ]:
backup("f5_b4_ativacao_init")

## Fase 5 · Bloco 5 — Função de erro do treino

**Pergunta:** Com qual função de erro a rede aprende melhor?

MSE (o padrão), MAE, Huber com três δ (quadrática perto de zero, linear nos extremos: menos sensível aos dias de movimento extremo), log-cosh (parecida com Huber, mas suave) e uma perda direcional: MSE + λ·média(relu(−ŷ·y)), que penaliza prever o sinal errado. Todas rodam os 5 folds.

**Base:** o melhor campeão entre `f5_b4_ativacao_init`.

| Eixo | Valores |
|---|---|
| `perda` | **mse** (`loss_fn=mse`), **mae** (`loss_fn=mae`), **huber δ=0,5** (`loss_fn=huber`, `huber_delta=0.5`), **huber δ=1** (`loss_fn=huber`, `huber_delta=1`), **huber δ=2** (`loss_fn=huber`, `huber_delta=2`), **logcosh** (`loss_fn=logcosh`), **direcional λ=0,1** (`loss_fn=direcional`, `loss_lambda=0.1`), **direcional λ=0,5** (`loss_fn=direcional`, `loss_lambda=0.5`), **direcional λ=1** (`loss_fn=direcional`, `loss_lambda=1`) |

**9 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** O efeito na métrica de decisão e, principalmente, na POCID: a perda direcional deveria melhorar o acerto de direção.

In [ ]:
rep.show_champion("f5_b4_ativacao_init")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_b5_perda.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f5_b5_perda")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_b5_perda")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_b5_perda`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_b5_perda")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f5_b5_perda")

#### Confirmação das finalistas e campeão — `f5_b5_perda`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_b5_perda", "final")

In [ ]:
rep.show_champion("f5_b5_perda")
rep.plot_finalists("f5_b5_perda")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b5_perda"), "f5_b5_perda__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b5_perda"), "f5_b5_perda__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bloco 5

- **Alguma função de erro bateu o MSE além do ruído?** _…_
- **A perda direcional melhorou a POCID?** _…_
- **Huber: qual δ?** _…_

In [ ]:
backup("f5_b5_perda")

## Fase 5 · Bloco 6 — Otimização

**Pergunta:** Qual algoritmo, taxa de aprendizagem, agenda da taxa, momentum e batch?

SGD, Adam, AdamW e RMSprop; taxa de aprendizagem de 0,0001 a 0,01; agenda da taxa (sem agenda, decaimento exponencial 0,99/0,97/0,95, redução no platô, cosseno); momentum; tamanho de batch. Espaço de 4 × 5 × 6 × 3 × 5 = 1.800 combinações, com busca aleatória de 60.

**Base:** o melhor campeão entre `f5_b5_perda`.

| Eixo | Valores |
|---|---|
| `optimizer` | `sgd`, `adam`, `adamw`, `rmsprop` |
| `lr` | `0.0001`, `0.0003`, `0.001`, `0.003`, `0.01` |
| `agenda` | **sem agenda** (`scheduler=none`), **exp 0,99** (`scheduler=exponential`, `decay_rate=0.99`), **exp 0,97** (`scheduler=exponential`, `decay_rate=0.97`), **exp 0,95** (`scheduler=exponential`, `decay_rate=0.95`), **platô** (`scheduler=plateau`), **cosseno** (`scheduler=cosine`) |
| `momentum` | `0`, `0.5`, `0.9` |
| `batch_size` | `16`, `32`, `64`, `128`, `256` |

**1800 combinações possíveis; busca aleatória de 60** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** A faixa útil de lr de cada otimizador e se a agenda da taxa importa.

In [ ]:
rep.show_champion("f5_b5_perda")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_b6_otimizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f5_b6_otimizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f5_b6_otimizacao", "triagem")

In [ ]:
rep.discarded_configs("f5_b6_otimizacao")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_b6_otimizacao")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_b6_otimizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_b6_otimizacao")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f5_b6_otimizacao", row="optimizer", col="lr")

In [ ]:
rep.heatmap("f5_b6_otimizacao", row="agenda", col="lr")

In [ ]:
rep.param_effect("f5_b6_otimizacao")

In [ ]:
rep.plot_grid_bars("f5_b6_otimizacao")

#### Confirmação das finalistas e campeão — `f5_b6_otimizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_b6_otimizacao", "final")

In [ ]:
rep.show_champion("f5_b6_otimizacao")
rep.plot_finalists("f5_b6_otimizacao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b6_otimizacao"), "f5_b6_otimizacao__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b6_otimizacao"), "f5_b6_otimizacao__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bloco 6

- **Otimizador e taxa:** _…_
- **Agenda da taxa de aprendizagem:** _…_
- **Momentum e batch:** _…_

In [ ]:
backup("f5_b6_otimizacao")

## Fase 5 · Checagem de interação

**Pergunta:** Com a otimização nova, a arquitetura campeã continua a melhor?

O 2º e o 3º colocados do bloco de arquitetura, treinados com a ativação, a perda e a otimização campeãs.

**Base:** o melhor campeão entre `f5_b6_otimizacao`.

Todas as configurações rodam os 5 folds.

**O que observar:** Se a ordem das arquiteturas se inverte.

In [ ]:
rep.show_champion("f5_b6_otimizacao")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_checagem.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f5_checagem")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_checagem")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### Confirmação das finalistas e campeão — `f5_checagem`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_checagem", "final")

In [ ]:
rep.show_champion("f5_checagem")
rep.plot_finalists("f5_checagem")

In [ ]:
pd.concat([rep.grid_ranking("f5_b6_otimizacao", "final").head(1),
           rep.grid_ranking("f5_checagem", "final")], ignore_index=True)

### 📝 Análise — Fase 5 · Checagem de interação

- **A ordem das arquiteturas se manteve?** _…_

In [ ]:
backup("f5_checagem")

## Fase 5 · Bloco 7 — Regularização: todos os dropouts

**Pergunta:** Quanto e onde aplicar dropout, e quanta penalização e corte de gradiente?

Todas as formas de dropout em LSTM: depois da recorrência e entre as densas (`dropout`), entre camadas empilhadas (`rnn_dropout`), na entrada (`input_dropout`) e **recorrente** (`recurrent_dropout`: no estado oculto entre um passo de tempo e o seguinte, com a mesma máscara em toda a sequência, como no Keras). Mais o decaimento dos pesos (L2) e o corte da norma do gradiente. Espaço de 5 × 3 × 3 × 4 × 4 × 3 = 2.160 combinações, com busca aleatória de 60; mais épocas e paciência.

**Base:** o melhor campeão entre `f5_b6_otimizacao`, `f5_checagem`.

| Eixo | Valores |
|---|---|
| `dropout` | `0`, `0.1`, `0.2`, `0.3`, `0.5` |
| `rnn_dropout` | `0`, `0.2`, `0.4` |
| `input_dropout` | `0`, `0.1`, `0.2` |
| `recurrent_dropout` | `0`, `0.1`, `0.2`, `0.3` |
| `weight_decay` | `0`, `1e-05`, `0.0001`, `0.001` |
| `grad_clip` | `0.5`, `1`, `5` |

**2160 combinações possíveis; busca aleatória de 60** (seed 0). Fixos no bloco: `epochs=200`, `patience=20`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O gap treino–validação, e se o dropout recorrente (o que faltava) ajuda.

In [ ]:
rep.show_champion("f5_b6_otimizacao")
rep.show_champion("f5_checagem")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_b7_regularizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f5_b7_regularizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f5_b7_regularizacao", "triagem")

In [ ]:
rep.discarded_configs("f5_b7_regularizacao")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_b7_regularizacao")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_b7_regularizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_b7_regularizacao")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f5_b7_regularizacao", row="dropout", col="recurrent_dropout")

In [ ]:
rep.heatmap("f5_b7_regularizacao", row="dropout", col="recurrent_dropout", value="gap/rmse_mean")

In [ ]:
rep.param_effect("f5_b7_regularizacao")

In [ ]:
rep.plot_grid_bars("f5_b7_regularizacao")

#### Confirmação das finalistas e campeão — `f5_b7_regularizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_b7_regularizacao", "final")

In [ ]:
rep.show_champion("f5_b7_regularizacao")
rep.plot_finalists("f5_b7_regularizacao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b7_regularizacao"), "f5_b7_regularizacao__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b7_regularizacao"), "f5_b7_regularizacao__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bloco 7

- **Qual dropout ajudou (se algum)?** _…_
- **Dropout recorrente:** _…_
- **Weight decay e corte de gradiente:** _…_
- **Gap e subajuste:** _…_

In [ ]:
backup("f5_b7_regularizacao")

## Fase 5 · Bloco 8 — Número de épocas

**Pergunta:** O resultado está limitado pelo orçamento de épocas? Quanta paciência o early stopping deve ter?

O número de épocas é decidido pelo early stopping na `val/loss`, e os pesos da melhor época são restaurados. O limite sobe para 500 épocas e varia só a paciência. Com decaimento da lr, paciência longa também dá tempo para a lr cair. O campeão deste bloco é o **resultado principal** do estudo.

**Base:** o melhor campeão entre `f5_b7_regularizacao`.

| Eixo | Valores |
|---|---|
| `patience` | `5`, `10`, `20`, `40` |

**4 combinações.** Fixos no bloco: `epochs=500`. A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Se a `melhor_epoca_media` encosta no limite e se paciência maior melhora a validação ou só gasta GPU.

In [ ]:
rep.show_champion("f5_b7_regularizacao")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_b8_epocas.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f5_b8_epocas")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_b8_epocas")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_b8_epocas`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_b8_epocas")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f5_b8_epocas")

#### Confirmação das finalistas e campeão — `f5_b8_epocas`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_b8_epocas", "final")

In [ ]:
rep.show_champion("f5_b8_epocas")
rep.plot_finalists("f5_b8_epocas")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b8_epocas"), "f5_b8_epocas__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_b8_epocas"), "f5_b8_epocas__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bloco 8

- **Paciência maior ajudou? O resultado estava limitado pelas épocas?** _…_
- **Custo (tempo) × ganho:** _…_

In [ ]:
backup("f5_b8_epocas")

## Fase 5 · Bônus — Data augmentation

**Pergunta:** Criar variações plausíveis das janelas de treino (ruído, amplitude, velocidade, ordem, recorte) melhora a previsão do campeão?

Data augmentation é **pré-processamento**, não hiperparâmetro da rede, por isso fica fora da sequência principal (como no estudo da CNN). A cada época, cada janela de treino é transformada com probabilidade 50%; validação e teste usam as janelas originais. As janelas estão normalizadas (z-score), então as intensidades são em desvios padrão. **Jittering:** ruído gaussiano (σ 0,03 fraca / 0,1 forte). **Scaling:** multiplica a janela por um fator ~ N(1, σ) (σ 0,1 / 0,2), simulando regimes de volatilidade; com alvo em retorno, o alvo é escalado junto. **Magnitude warping:** multiplica por uma curva suave aleatória (σ 0,1 / 0,2). **Time warping:** acelera e desacelera trechos da janela (σ 0,1 / 0,2), preservando o dia da decisão. **Permutation:** embaralha 3 ou 6 segmentos da janela; se não piorar, o modelo não está usando a ordem temporal. **Window slicing:** recorta 90% ou 70% da janela e reamostra para o tamanho original (a janela deslizante com sobreposição já é como as amostras são montadas). Também entra a combinação clássica jitter + scaling. **Base:** o campeão principal, re-treinado aqui sem augmentation como referência pareada, com mais épocas e paciência para todos, porque augmentation retarda a convergência. Em séries financeiras o ganho não é garantido; o bloco mede quais estratégias ajudam e quais atrapalham.

**Base:** o melhor campeão entre `f5_b8_epocas`.

| Eixo | Valores |
|---|---|
| `augment` | `none`, `jitter`, `scaling`, `magwarp`, `timewarp`, `permutation`, `slicing`, `jitter+scaling` |
| `aug_strength` | `fraca`, `forte` |

**16 combinações.** Fixos no bloco: `epochs=300`, `patience=30`, `aug_prob=0.5`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O ganho pareado sobre a referência sem augmentation, a queda do gap treino–validação, e se a permutação piora (o que confirma que o LSTM usa a ordem temporal).

In [ ]:
rep.show_champion("f5_b8_epocas")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_bonus_augmentation.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f5_bonus_augmentation`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f5_bonus_augmentation", "triagem")

In [ ]:
rep.discarded_configs("f5_bonus_augmentation")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f5_bonus_augmentation")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f5_bonus_augmentation`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f5_bonus_augmentation")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f5_bonus_augmentation", row="augment", col="aug_strength")

In [ ]:
rep.heatmap("f5_bonus_augmentation", row="augment", col="aug_strength", value="gap/rmse_mean")

In [ ]:
rep.heatmap("f5_bonus_augmentation", row="augment", col="aug_strength", value="val/pocid_mean")

In [ ]:
rep.plot_grid_bars("f5_bonus_augmentation")

#### Confirmação das finalistas e campeão — `f5_bonus_augmentation`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f5_bonus_augmentation", "final")

In [ ]:
rep.show_champion("f5_bonus_augmentation")
rep.plot_finalists("f5_bonus_augmentation")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_bonus_augmentation"), "f5_bonus_augmentation__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f5_bonus_augmentation"), "f5_bonus_augmentation__*", metric="val/pocid")

### 📝 Análise — Fase 5 · Bônus

- **Alguma estratégia melhorou a validação além do ruído entre seeds?** _…_
- **Qual estratégia reduziu mais o gap treino–validação?** _…_
- **Permutação: o modelo depende da ordem temporal?** _…_
- **Time warping e slicing: distorcer o tempo ajuda ou atrapalha em retornos diários?** _…_
- **Intensidade fraca × forte:** _…_

In [ ]:
backup("f5_bonus_augmentation")

## Fase 5 · Ablação do campeão

**Pergunta:** Das mudanças que levaram do LSTM padrão ao campeão, quais realmente melhoram o modelo e quais são dispensáveis?

Cada configuração parte do campeão principal e **desfaz uma única mudança**, voltando aquele hiperparâmetro ao valor do padrão do estudo. Todas rodam os K folds e são comparadas fold a fold com o campeão re-treinado aqui. Se piora ao desfazer (além do ruído entre seeds), a mudança ajuda; se fica dentro do ruído, é dispensável; se melhora ao desfazer, a mudança atrapalhava e só entrou por sorte na seleção. Hiperparâmetros cuja reversão não muda nada (ex.: momentum com Adam) são descartados como equivalentes. É a resposta mais direta para "o que melhora o modelo quando mexemos".

**Base:** o melhor campeão entre `f5_b8_epocas`.

A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Quais mudanças ficam verdes (ajudam além do ruído) e se as mais importantes batem com a importância dos eixos em cada bloco.

In [ ]:
rep.show_champion("f5_b8_epocas")

In [ ]:
if executar("fase5"):
    !python src/grid_search.py grids/f5_ablacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultado da ablação — `f5_ablacao`

Δ = (campeão sem a mudança) − (campeão), fold a fold. Verde: a mudança ajuda além do ruído entre seeds; cinza: dispensável; vermelho: atrapalhava.

In [ ]:
rep.discarded_configs("f5_ablacao")  # mudanças cuja reversão não altera nada

In [ ]:
rep.ablation_table("f5_ablacao")

### 📝 Análise — Fase 5 · Ablação do campeão

- **Quais mudanças realmente ajudam (além do ruído)?** _…_
- **Quais são dispensáveis e poderiam voltar ao padrão?** _…_
- **Alguma mudança atrapalhava?** _…_
- **Isso confirma a importância dos eixos vista em cada bloco?** _…_

In [ ]:
backup("f5_ablacao")

## Fase 5 · O que realmente melhorou o modelo (validação)

A **cadeia de decisões** reconstrói, pela herança entre blocos, o caminho do LSTM padrão até o campeão (`f5_b8_epocas`): o que
mudou em cada passo, o Δ pareado por fold, em quantos folds a mudança venceu e se o ganho passa do ruído entre seeds.

In [ ]:
cadeia = rep.decision_chain("f5_b8_epocas")
cadeia

## Fase 5 · Teste revelado

Até aqui todas as escolhas desta fase usaram só a validação. Agora o período de teste é usado **uma única vez**, para as
referências e o campeão de cada bloco (média ± desvio entre os modelos dos K folds). A concordância entre validação e
teste é a evidência de que não houve vazamento; uma discrepância grande (ex.: mudança de regime) é um achado.

In [ ]:
final = rep.final_report(["f5_referencia_modelos", "f5_b1_entrada", "f5_b2_normalizacao", "f5_b3_arquitetura", "f5_b4_ativacao_init", "f5_b5_perda", "f5_b6_otimizacao", "f5_checagem", "f5_b7_regularizacao", "f5_b8_epocas", "f5_bonus_augmentation"])
final

In [ ]:
campeao_fase = rep.champion_name("f5_b8_epocas")
rep.plot_predictions(campeao_fase)

### Fusão de modelos

Inspirada na Combinatorial Fusion Analysis do paper de Wu et al. (2025): todas as combinações de 2 a N modelos, com
pesos iguais, por desempenho (1/MSE de validação) ou por **diversidade cognitiva** (distância entre as funções
rank-score dos modelos), combinando valores previstos (escore) ou posições (rank). Diferente do paper, os pesos vêm só da
validação walk-forward (cada dia previsto por um modelo que ainda não o viu) e a fusão é **escolhida uma vez, pela
validação**, antes de olhar o teste; o paper escolhia a cada dia a combinação mais próxima do preço real.

In [ ]:
modelos_fusao = list(rep.results.configs("f5_referencia_modelos")["exp_name"]) + [rep.champion_name(b) for b in ["f5_b8_epocas", "f5_bonus_augmentation"] if rep.champion_name(b)]
fusao = rep.fusion_report(modelos_fusao, max_size=5)
fusao.head(15)

In [ ]:
wandb_report.log_study("f5_b8_epocas", ["f5_referencia_modelos", "f5_b1_entrada", "f5_b2_normalizacao", "f5_b3_arquitetura", "f5_b4_ativacao_init", "f5_b5_perda", "f5_b6_otimizacao", "f5_checagem", "f5_b7_regularizacao", "f5_b8_epocas", "f5_bonus_augmentation"], nome="fase5__resumo")
backup("fase5")

### O que descobrimos nesta fase

Na execução de referência:

- Ainda não executada: esta fase é a próxima a rodar (os números aparecem nas células de análise).
- No teste funcional (2 folds, 3 épocas), todos os componentes novos treinaram; a receita do paper com alvo em preço extrapolou (Theil 498), como o alvo em preço na fase 1.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Fase 6 — Lacunas da fase 5: ativações, dropout recorrente, horizonte e TimeGAN

**Motivação.** Na fase 5, o campeão de arquitetura saiu bidirecional e a célula LSTM própria não suportava bidirecional: 72 combinações de ativações e todas as 160 com dropout recorrente foram descartadas. A célula própria agora é bidirecional, e os dois blocos são refeitos a partir do campeão da fase 5. A fase também mede o efeito do horizonte de previsão (1, 5 e 20 dias) e, como bônus, dados sintéticos com TimeGAN, com diagnósticos para saber se o gerador aprende o BTC ou só o ruído.

**Blocos desta fase:**

- `f6_b1_ativacao_init` — Fase 6 · Ativações e inicialização (refeito): Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã? (lstm_activation × activation × weight_init (aleatória, 40))
- `f6_b2_regularizacao` — Fase 6 · Regularização com dropout recorrente (refeito): Quanto e onde aplicar dropout, e quanta penalização e corte de gradiente? (dropout × rnn_dropout × input_dropout × recurrent_dropout × weight_decay × grad_clip (aleatória, 60))
- `f6_horizonte` — Fase 6 · Horizonte de previsão: Prever o preço de 5 ou 20 dias à frente é mais fácil que prever o de amanhã? (horizon)
- `f6_bonus_timegan` — Fase 6 · Bônus — Dados sintéticos com TimeGAN (série longa): Janelas sintéticas geradas por um TimeGAN ajudam o LSTM, ou o gerador só reproduz o ruído? (gan)

**Série:** `data/btc-usd_yahoo_2014-09-17_2026-09-23.csv`, de 2014-09-17 a 2026-09-23; 5 folds de validação de 365 dias; teste a partir de 2025-09-24.

In [ ]:
usar_estudo("config/estudo_btc_longo.json")
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())
!python tests/check_data.py
!python tests/check_features.py

## Fase 6 · Ativações e inicialização (refeito)

**Pergunta:** Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã?

Refaz o bloco de ativações da fase 5, que perdeu 72 das combinações: o campeão de arquitetura é bidirecional e a célula LSTM própria (usada para ativações ≠ tanh) não suportava bidirecional. Agora suporta (a direção reversa reproduz o nn.LSTM bidirecional com diferença de 1e-7). Itens 4 e 6 da lista. **Célula LSTM:** as portas são sempre sigmoid; varia a ativação da candidata e da saída (tanh é o padrão; sigmoid, softsign e ReLU usam uma célula própria, mais lenta que o cuDNN). **Densas:** ReLU, tanh, sigmoid e ELU; se a campeã não tiver camada densa, esse eixo não tem efeito e as combinações equivalentes são descartadas. A saída é sempre linear, porque o alvo é contínuo (sigmoid e softmax na saída são para classificação). **Inicialização:** pesos pequenos e aleatórios; `padrao` = PyTorch U(±1/√h), `uniforme` = U(±0,05), `normal` = N(0; 0,05), `xavier` (Glorot), `glorot_ortogonal` (padrão do Keras: ortogonal na recorrência e bias de esquecimento 1) e `he` (Kaiming, pensado para ReLU). Espaço de 4 × 4 × 6 = 96 combinações, com busca aleatória de 40.

**Base:** o melhor campeão entre `f5_b8_epocas`.

| Eixo | Valores |
|---|---|
| `lstm_activation` | `tanh`, `sigmoid`, `softsign`, `relu` |
| `activation` | `relu`, `tanh`, `sigmoid`, `elu` |
| `weight_init` | `padrao`, `uniforme`, `normal`, `xavier`, `glorot_ortogonal`, `he` |

**96 combinações possíveis; busca aleatória de 40** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** Se a ativação da célula muda algo além de tanh (ReLU pode explodir sem limite), e se a inicialização afeta a convergência (`melhor_epoca_media`) e a variância entre folds.

In [ ]:
rep.show_champion("f5_b8_epocas")

In [ ]:
if executar("fase6"):
    !python src/grid_search.py grids/f6_b1_ativacao_init.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f6_b1_ativacao_init`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f6_b1_ativacao_init", "triagem")

In [ ]:
rep.discarded_configs("f6_b1_ativacao_init")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f6_b1_ativacao_init")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f6_b1_ativacao_init`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f6_b1_ativacao_init")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f6_b1_ativacao_init", row="weight_init", col="lstm_activation")

In [ ]:
rep.heatmap("f6_b1_ativacao_init", row="weight_init", col="lstm_activation", value="melhor_epoca_media")

In [ ]:
rep.heatmap("f6_b1_ativacao_init", row="activation", col="lstm_activation")

In [ ]:
rep.param_effect("f6_b1_ativacao_init")

In [ ]:
rep.plot_grid_bars("f6_b1_ativacao_init")

#### Confirmação das finalistas e campeão — `f6_b1_ativacao_init`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f6_b1_ativacao_init", "final")

In [ ]:
rep.show_champion("f6_b1_ativacao_init")
rep.plot_finalists("f6_b1_ativacao_init")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f6_b1_ativacao_init"), "f6_b1_ativacao_init__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f6_b1_ativacao_init"), "f6_b1_ativacao_init__*", metric="val/pocid")

### 📝 Análise — Fase 6 · Ativações e inicialização (refeito)

- **Ativação da célula LSTM:** _…_
- **Ativação das densas:** _…_
- **Inicialização: afetou a convergência ou só o ruído entre folds?** _…_
- **Ganho sobre o Bloco 2 (validação):** _…_

In [ ]:
backup("f6_b1_ativacao_init")

## Fase 6 · Regularização com dropout recorrente (refeito)

**Pergunta:** Quanto e onde aplicar dropout, e quanta penalização e corte de gradiente?

Refaz o bloco de regularização da fase 5, em que todas as 160 combinações com dropout recorrente foram descartadas pelo mesmo motivo (célula própria sem suporte a bidirecional). Todas as formas de dropout em LSTM: depois da recorrência e entre as densas (`dropout`), entre camadas empilhadas (`rnn_dropout`), na entrada (`input_dropout`) e **recorrente** (`recurrent_dropout`: no estado oculto entre um passo de tempo e o seguinte, com a mesma máscara em toda a sequência, como no Keras). Mais o decaimento dos pesos (L2) e o corte da norma do gradiente. Espaço de 5 × 3 × 3 × 4 × 4 × 3 = 2.160 combinações, com busca aleatória de 60; mais épocas e paciência.

**Base:** o melhor campeão entre `f6_b1_ativacao_init`.

| Eixo | Valores |
|---|---|
| `dropout` | `0`, `0.1`, `0.2`, `0.3`, `0.5` |
| `rnn_dropout` | `0`, `0.2`, `0.4` |
| `input_dropout` | `0`, `0.1`, `0.2` |
| `recurrent_dropout` | `0`, `0.1`, `0.2`, `0.3` |
| `weight_decay` | `0`, `1e-05`, `0.0001`, `0.001` |
| `grad_clip` | `0.5`, `1`, `5` |

**2160 combinações possíveis; busca aleatória de 60** (seed 0). Fixos no bloco: `epochs=200`, `patience=20`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O gap treino–validação, e se o dropout recorrente (o que faltava) ajuda.

In [ ]:
rep.show_champion("f6_b1_ativacao_init")

In [ ]:
if executar("fase6"):
    !python src/grid_search.py grids/f6_b2_regularizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

#### Resultados da triagem — `f6_b2_regularizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("f6_b2_regularizacao", "triagem")

In [ ]:
rep.discarded_configs("f6_b2_regularizacao")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f6_b2_regularizacao")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f6_b2_regularizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f6_b2_regularizacao")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("f6_b2_regularizacao", row="dropout", col="recurrent_dropout")

In [ ]:
rep.heatmap("f6_b2_regularizacao", row="dropout", col="recurrent_dropout", value="gap/rmse_mean")

In [ ]:
rep.param_effect("f6_b2_regularizacao")

In [ ]:
rep.plot_grid_bars("f6_b2_regularizacao")

#### Confirmação das finalistas e campeão — `f6_b2_regularizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f6_b2_regularizacao", "final")

In [ ]:
rep.show_champion("f6_b2_regularizacao")
rep.plot_finalists("f6_b2_regularizacao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f6_b2_regularizacao"), "f6_b2_regularizacao__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f6_b2_regularizacao"), "f6_b2_regularizacao__*", metric="val/pocid")

### 📝 Análise — Fase 6 · Regularização com dropout recorrente (refeito)

- **Qual dropout ajudou (se algum)?** _…_
- **Dropout recorrente:** _…_
- **Weight decay e corte de gradiente:** _…_
- **Gap e subajuste:** _…_

In [ ]:
backup("f6_b2_regularizacao")

## Fase 6 · Horizonte de previsão

**Pergunta:** Prever o preço de 5 ou 20 dias à frente é mais fácil que prever o de amanhã?

O campeão de regularização com horizonte de 1, 5 e 20 dias: o alvo passa a ser o log-retorno de h dias (o preço de t+h), e a referência é o passeio aleatório no mesmo horizonte (prever que o preço de t+h é o de t). O embargo entre treino e validação cresce junto (h dias). O RMSE de horizontes diferentes não é comparável (retornos maiores, erros maiores), por isso este bloco decide pelo **Theil**, que já é relativo ao passeio aleatório do mesmo horizonte. Com h > 1, dias vizinhos compartilham parte do período previsto, então a POCID e a acurácia direcional ficam correlacionadas entre dias.

**Base:** o melhor campeão entre `f6_b2_regularizacao`.

| Eixo | Valores |
|---|---|
| `horizon` | `1`, `5`, `20` |

**3 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Se o Theil cai (e a POCID sobe) com horizontes maiores, onde há menos ruído relativo.

In [ ]:
rep.show_champion("f6_b2_regularizacao")

In [ ]:
if executar("fase6"):
    !python src/grid_search.py grids/f6_horizonte.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f6_horizonte")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f6_horizonte")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f6_horizonte`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f6_horizonte", metric="val/theil")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f6_horizonte")

#### Confirmação das finalistas e campeão — `f6_horizonte`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f6_horizonte", "final")

In [ ]:
rep.show_champion("f6_horizonte")
rep.plot_finalists("f6_horizonte")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f6_horizonte"), "f6_horizonte__*", metric="val/theil")

In [ ]:
rep.paired_comparison(rep.base_config_name("f6_horizonte"), "f6_horizonte__*", metric="val/pocid")

### 📝 Análise — Fase 6 · Horizonte de previsão

- **Com qual horizonte o modelo mais se afasta do passeio aleatório?** _…_
- **A direção (POCID, acurácia direcional) melhora em horizontes maiores?** _…_

In [ ]:
backup("f6_horizonte")

## Fase 6 · Bônus — Dados sintéticos com TimeGAN (série longa)

**Pergunta:** Janelas sintéticas geradas por um TimeGAN ajudam o LSTM, ou o gerador só reproduz o ruído?

TimeGAN (Yoon et al., 2019) treinado dentro de cada fold, só com as janelas de treino (a janela e o alvo juntos, como uma sequência), com 1.000 ou 3.000 iterações por etapa. **Mistura:** a cada batch de treino, janelas sintéticas são somadas às reais na proporção indicada. **Só sintético (TSTR, train on synthetic, test on real):** o LSTM treina só com o sintético e é avaliado na validação real; se ficar muito pior que treinar com dados reais, o gerador aprendeu ruído, não o BTC. **Diagnósticos por fold:** um classificador tenta separar janelas reais de sintéticas (0 = indistinguíveis), e os fatos estilizados do retorno (curtose, autocorrelação do retorno e do |retorno|) são comparados entre real e sintético. O risco apontado no desenho do experimento é que, numa série tão ruidosa e movida por eventos externos, o gerador reproduza o ruído e não as propriedades do BTC; os diagnósticos medem isso diretamente.

**Base:** o melhor campeão entre `f6_b2_regularizacao`.

| Eixo | Valores |
|---|---|
| `gan` | **sem TimeGAN** (`timegan=none`), **mistura 0,5×** (`timegan=mistura`, `timegan_ratio=0.5`, `timegan_iter=1000`), **mistura 1×** (`timegan=mistura`, `timegan_ratio=1`, `timegan_iter=1000`), **mistura 2×** (`timegan=mistura`, `timegan_ratio=2`, `timegan_iter=1000`), **mistura 1× (3000 it)** (`timegan=mistura`, `timegan_ratio=1`, `timegan_iter=3000`), **só sintético (TSTR)** (`timegan=so_sintetico`, `timegan_iter=1000`), **só sintético (TSTR, 3000 it)** (`timegan=so_sintetico`, `timegan_iter=3000`) |

**7 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Os diagnósticos do gerador (fatos estilizados, discriminativo), o TSTR e o Δ pareado da mistura sobre o treino só com dados reais.

In [ ]:
rep.show_champion("f6_b2_regularizacao")

In [ ]:
if executar("fase6"):
    !python src/grid_search.py grids/f6_bonus_timegan.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f6_bonus_timegan")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f6_bonus_timegan")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f6_bonus_timegan`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f6_bonus_timegan")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f6_bonus_timegan")

#### Confirmação das finalistas e campeão — `f6_bonus_timegan`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f6_bonus_timegan", "final")

In [ ]:
rep.show_champion("f6_bonus_timegan")
rep.plot_finalists("f6_bonus_timegan")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f6_bonus_timegan"), "f6_bonus_timegan__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f6_bonus_timegan"), "f6_bonus_timegan__*", metric="val/pocid")

#### Diagnósticos do TimeGAN

O gerador reproduz as propriedades do BTC ou só o ruído? Fatos estilizados do retorno (real × sintético), score discriminativo e o TSTR (linha "só sintético").

In [ ]:
rep.timegan_diagnostics("f6_bonus_timegan")

### 📝 Análise — Fase 6 · Bônus

- **O gerador reproduz as caudas pesadas e os aglomerados de volatilidade do BTC?** _…_
- **TSTR: o LSTM treinado só com sintético prevê o BTC real?** _…_
- **A mistura melhorou a validação além do ruído?** _…_
- **Mais iterações do TimeGAN melhoraram o gerador?** _…_

In [ ]:
backup("f6_bonus_timegan")

## Fase 6 · O que realmente melhorou o modelo (validação)

A **cadeia de decisões** reconstrói, pela herança entre blocos, o caminho do LSTM padrão até o campeão (`f6_b2_regularizacao`): o que
mudou em cada passo, o Δ pareado por fold, em quantos folds a mudança venceu e se o ganho passa do ruído entre seeds.

In [ ]:
cadeia = rep.decision_chain("f6_b2_regularizacao")
cadeia

## Fase 6 · Teste revelado

Até aqui todas as escolhas desta fase usaram só a validação. Agora o período de teste é usado **uma única vez**, para as
referências e o campeão de cada bloco (média ± desvio entre os modelos dos K folds). A concordância entre validação e
teste é a evidência de que não houve vazamento; uma discrepância grande (ex.: mudança de regime) é um achado.

In [ ]:
final = rep.final_report(["f6_b1_ativacao_init", "f6_b2_regularizacao", "f6_horizonte", "f6_bonus_timegan"])
final

In [ ]:
campeao_fase = rep.champion_name("f6_b2_regularizacao")
rep.plot_predictions(campeao_fase)

### O que descobrimos nesta fase

Na execução de referência:

- Ainda não executada.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Fase 7 — Mais dados reais: várias criptomoedas no treino

**Motivação.** A série longa (fase 3) foi a única mudança que moveu o teste: mais dado real. Esta fase multiplica os exemplos de treino com as janelas de outras 14 criptomoedas (ETH, LTC, XRP, BNB, DOGE, BCH, XLM, ADA, LINK, TRX, SOL, XMR, DASH, ETC), num único LSTM, avaliado sempre no BTC. É o análogo, nesta série, do data augmentation da CNN, com dados reais em vez de transformados. Como bônus, o TimeGAN sobre o conjunto.

**Blocos desta fase:**

- `f7_referencia` — Fase 7 · Referência (só BTC, mesmas datas): O estudo com várias moedas reproduz a série longa quando treina só com o BTC? (passeio_aleatorio, media_historica, ultimo_retorno, regressao_linear, lstm_padrao, lstm_padrao_seed1, lstm_padrao_seed2, lstm_padrao_seed3, lstm_padrao_seed4)
- `f7_ativos` — Fase 7 · Quantas criptomoedas no treino: Treinar o LSTM com janelas de outras criptomoedas melhora a previsão do BTC? (ativos)
- `f7_bonus_timegan` — Fase 7 · Bônus — TimeGAN sobre o conjunto de criptomoedas: Com muito mais janelas reais para aprender, o TimeGAN gera dados sintéticos mais úteis? (gan)

**Série:** `data/cripto/BTC.csv`, de 2014-09-17 a 2026-09-23; 5 folds de validação de 365 dias; teste a partir de 2025-09-24.

**Ressalva:** Mesmo ano de teste das fases 3 a 6.

In [ ]:
usar_estudo("config/estudo_multicripto.json")
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())
!python tests/check_data.py
!python tests/check_features.py

## Fase 7 · Referência (só BTC, mesmas datas)

**Pergunta:** O estudo com várias moedas reproduz a série longa quando treina só com o BTC?

As mesmas referências da série longa (passeio aleatório, média, último retorno, regressão linear e o LSTM padrão com 5 seeds), treinadas só com o BTC. Valida que este estudo reproduz a série longa antes de acrescentar outras moedas, e dá a régua de ruído entre seeds da fase.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Configuração | Parâmetros |
|---|---|
| `passeio_aleatorio` | `model=naive_zero` |
| `media_historica` | `model=naive_mean` |
| `ultimo_retorno` | `model=naive_last` |
| `regressao_linear` | `model=linear` |
| `lstm_padrao` | `model=lstm` |
| `lstm_padrao_seed1` | `model=lstm`, `seed=1` |
| `lstm_padrao_seed2` | `model=lstm`, `seed=2` |
| `lstm_padrao_seed3` | `model=lstm`, `seed=3` |
| `lstm_padrao_seed4` | `model=lstm`, `seed=4` |

**9 configurações.** Todas as configurações rodam os 5 folds.

**O que observar:** Se o Theil de teste do LSTM padrão fica abaixo de 1 em todas as seeds, e se a acurácia de direção sai de ~50%.

In [ ]:
if executar("fase7"):
    !python src/grid_search.py grids/f7_referencia.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA_B0

In [ ]:
rep.discarded_configs("f7_referencia")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f7_referencia")  # treinaram, mas divergiram (val/theil > limite): fora das médias

In [ ]:
rep.plot_grid_bars("f7_referencia")

#### Confirmação das finalistas e campeão — `f7_referencia`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f7_referencia", "final")

In [ ]:
rep.show_champion("f7_referencia")
rep.plot_finalists("f7_referencia")

#### Ruído entre seeds

A régua do estudo: o menor efeito que se distingue de sorte na inicialização.

In [ ]:
ruido = rep.noise_floor()
rep.noise_floor("val/pocid")

### 📝 Análise — Fase 7 · Referência (só BTC, mesmas datas)

- **Algum modelo bateu o passeio aleatório (theil < 1, skill > 0)?** _…_
- **O LSTM padrão superou a regressão linear na mesma janela?** _…_
- **POCID e acurácia direcional: acima de 50% de forma consistente?** _…_
- **Ruído entre seeds: qual o menor efeito que o estudo consegue detectar?** _…_

In [ ]:
backup("f7_referencia")

## Fase 7 · Quantas criptomoedas no treino

**Pergunta:** Treinar o LSTM com janelas de outras criptomoedas melhora a previsão do BTC?

O melhor LSTM da fase 6, treinado com as janelas de 1, 2, 5, 10 ou 15 criptomoedas (normalização por moeda; um único modelo). A validação e o teste são sempre no BTC. É o equivalente, nesta série, ao data augmentation da CNN: mais exemplos de treino, só que reais: a dinâmica de curto prazo das criptomoedas é parecida, então as outras moedas mostram ao modelo muito mais situações de mercado. Moedas lançadas depois de 2017 só entram nos folds em que já existiam.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Eixo | Valores |
|---|---|
| `ativos` | `btc`, `btc_eth`, `top5`, `top10`, `todas` |

**5 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Se o Theil do BTC cai à medida que entram mais moedas, e a partir de quantas moedas satura (ou piora, se as outras moedas se comportarem diferente do BTC).

In [ ]:
if executar("fase7"):
    !python src/grid_search.py grids/f7_ativos.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f7_ativos")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f7_ativos")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f7_ativos`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f7_ativos")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f7_ativos")

#### Confirmação das finalistas e campeão — `f7_ativos`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f7_ativos", "final")

In [ ]:
rep.show_champion("f7_ativos")
rep.plot_finalists("f7_ativos")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f7_ativos"), "f7_ativos__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f7_ativos"), "f7_ativos__*", metric="val/pocid")

### 📝 Análise — Fase 7 · Quantas criptomoedas no treino

- **Mais moedas melhoraram a previsão do BTC além do ruído?** _…_
- **Com quantas moedas o ganho satura?** _…_
- **E a direção (POCID)?** _…_

In [ ]:
backup("f7_ativos")

## Fase 7 · Bônus — TimeGAN sobre o conjunto de criptomoedas

**Pergunta:** Com muito mais janelas reais para aprender, o TimeGAN gera dados sintéticos mais úteis?

TimeGAN (Yoon et al., 2019) treinado dentro de cada fold, só com as janelas de treino (a janela e o alvo juntos, como uma sequência), com 1.000 ou 3.000 iterações por etapa. **Mistura:** a cada batch de treino, janelas sintéticas são somadas às reais na proporção indicada. **Só sintético (TSTR, train on synthetic, test on real):** o LSTM treina só com o sintético e é avaliado na validação real; se ficar muito pior que treinar com dados reais, o gerador aprendeu ruído, não o BTC. **Diagnósticos por fold:** um classificador tenta separar janelas reais de sintéticas (0 = indistinguíveis), e os fatos estilizados do retorno (curtose, autocorrelação do retorno e do |retorno|) são comparados entre real e sintético. O risco apontado no desenho do experimento é que, numa série tão ruidosa e movida por eventos externos, o gerador reproduza o ruído e não as propriedades do BTC; os diagnósticos medem isso diretamente. Aqui o gerador aprende com as janelas de todas as moedas do grupo campeão (muito mais dados que na fase 6).

**Base:** o melhor campeão entre `f7_ativos`.

| Eixo | Valores |
|---|---|
| `gan` | **sem TimeGAN** (`timegan=none`), **mistura 1×** (`timegan=mistura`, `timegan_ratio=1`, `timegan_iter=1000`), **mistura 1× (3000 it)** (`timegan=mistura`, `timegan_ratio=1`, `timegan_iter=3000`), **só sintético (TSTR)** (`timegan=so_sintetico`, `timegan_iter=1000`) |

**4 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Se os diagnósticos do gerador melhoram em relação à fase 6 e se a mistura passa a ajudar.

In [ ]:
rep.show_champion("f7_ativos")

In [ ]:
if executar("fase7"):
    !python src/grid_search.py grids/f7_bonus_timegan.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f7_bonus_timegan")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f7_bonus_timegan")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f7_bonus_timegan`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f7_bonus_timegan")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f7_bonus_timegan")

#### Confirmação das finalistas e campeão — `f7_bonus_timegan`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f7_bonus_timegan", "final")

In [ ]:
rep.show_champion("f7_bonus_timegan")
rep.plot_finalists("f7_bonus_timegan")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("f7_bonus_timegan"), "f7_bonus_timegan__*")

In [ ]:
rep.paired_comparison(rep.base_config_name("f7_bonus_timegan"), "f7_bonus_timegan__*", metric="val/pocid")

#### Diagnósticos do TimeGAN

O gerador reproduz as propriedades do BTC ou só o ruído? Fatos estilizados do retorno (real × sintético), score discriminativo e o TSTR (linha "só sintético").

In [ ]:
rep.timegan_diagnostics("f7_bonus_timegan")

### 📝 Análise — Fase 7 · Bônus

- **Os diagnósticos do gerador melhoraram com mais dados?** _…_
- **A mistura ajudou o BTC?** _…_
- **TSTR:** _…_

In [ ]:
backup("f7_bonus_timegan")

## Fase 7 · O que realmente melhorou o modelo (validação)

A **cadeia de decisões** reconstrói, pela herança entre blocos, o caminho do LSTM padrão até o campeão (`f7_ativos`): o que
mudou em cada passo, o Δ pareado por fold, em quantos folds a mudança venceu e se o ganho passa do ruído entre seeds.

In [ ]:
cadeia = rep.decision_chain("f7_ativos")
cadeia

## Fase 7 · Teste revelado

Até aqui todas as escolhas desta fase usaram só a validação. Agora o período de teste é usado **uma única vez**, para as
referências e o campeão de cada bloco (média ± desvio entre os modelos dos K folds). A concordância entre validação e
teste é a evidência de que não houve vazamento; uma discrepância grande (ex.: mudança de regime) é um achado.

In [ ]:
final = rep.final_report(["f7_referencia", "f7_ativos", "f7_bonus_timegan"])
final

In [ ]:
campeao_fase = rep.champion_name("f7_ativos")
rep.plot_predictions(campeao_fase)

### O que descobrimos nesta fase

Na execução de referência:

- Ainda não executada.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Fase 8 — Mais dados reais: dados por hora

**Motivação.** Outra forma de ter mais dado real sobre o mesmo ativo: velas de hora em hora (Binance, desde 2017). O modelo continua prevendo o preço do dia seguinte e é avaliado nos mesmos pontos diários, mas treina com uma janela por hora (24× mais exemplos) e vê o que aconteceu dentro de cada dia.

**Blocos desta fase:**

- `f8_referencia` — Fase 8 · Referência com dados por hora: Com dados de hora em hora (24× mais janelas de treino), o LSTM padrão prevê melhor o preço do dia seguinte? (passeio_aleatorio, media_historica, ultimo_retorno, regressao_linear, lstm_padrao, lstm_padrao_seed1, lstm_padrao_seed2, lstm_padrao_seed3, lstm_padrao_seed4)
- `f8_janela` — Fase 8 · O melhor LSTM com dados por hora: tamanho da janela: Quantas horas de histórico o LSTM precisa ver, com dados por hora? (lookback)

**Série:** `data/btc-usdt_binance_1h_2017-08-17_2026-09-23.csv`, de 2017-08-17 a 2026-09-23; 5 folds de validação de 365 dias; teste a partir de 2025-09-24.

**Ressalva:** O treino começa em 2017-08 (início dos dados da Binance), e não em 2014 como na série longa.

In [ ]:
usar_estudo("config/estudo_btc_horario.json")
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())
!python tests/check_data.py
!python tests/check_features.py

## Fase 8 · Referência com dados por hora

**Pergunta:** Com dados de hora em hora (24× mais janelas de treino), o LSTM padrão prevê melhor o preço do dia seguinte?

Passeio aleatório, média, último retorno (as últimas 24 horas), regressão linear e o LSTM padrão com 5 seeds, agora com janelas de 168 horas (7 dias) e alvo 24 horas à frente, avaliados só às 00:00 UTC: exatamente os mesmos pontos e o mesmo alvo diário da série longa, com 24× mais janelas de treino (uma por hora). As métricas são diretamente comparáveis às da fase 3.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Configuração | Parâmetros |
|---|---|
| `passeio_aleatorio` | `model=naive_zero` |
| `media_historica` | `model=naive_mean` |
| `ultimo_retorno` | `model=naive_last` |
| `regressao_linear` | `model=linear` |
| `lstm_padrao` | `model=lstm` |
| `lstm_padrao_seed1` | `model=lstm`, `seed=1` |
| `lstm_padrao_seed2` | `model=lstm`, `seed=2` |
| `lstm_padrao_seed3` | `model=lstm`, `seed=3` |
| `lstm_padrao_seed4` | `model=lstm`, `seed=4` |

**9 configurações.** Todas as configurações rodam os 5 folds.

**O que observar:** Se o Theil de teste do LSTM padrão fica abaixo de 1 em todas as seeds, e se a acurácia de direção sai de ~50%.

In [ ]:
if executar("fase8"):
    !python src/grid_search.py grids/f8_referencia.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA_B0

In [ ]:
rep.discarded_configs("f8_referencia")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f8_referencia")  # treinaram, mas divergiram (val/theil > limite): fora das médias

In [ ]:
rep.plot_grid_bars("f8_referencia")

#### Confirmação das finalistas e campeão — `f8_referencia`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f8_referencia", "final")

In [ ]:
rep.show_champion("f8_referencia")
rep.plot_finalists("f8_referencia")

#### Ruído entre seeds

A régua do estudo: o menor efeito que se distingue de sorte na inicialização.

In [ ]:
ruido = rep.noise_floor()
rep.noise_floor("val/pocid")

### 📝 Análise — Fase 8 · Referência com dados por hora

- **Algum modelo bateu o passeio aleatório (theil < 1, skill > 0)?** _…_
- **O LSTM padrão superou a regressão linear na mesma janela?** _…_
- **POCID e acurácia direcional: acima de 50% de forma consistente?** _…_
- **Ruído entre seeds: qual o menor efeito que o estudo consegue detectar?** _…_

In [ ]:
backup("f8_referencia")

## Fase 8 · O melhor LSTM com dados por hora: tamanho da janela

**Pergunta:** Quantas horas de histórico o LSTM precisa ver, com dados por hora?

O melhor LSTM da fase 6 levado para os dados por hora, com janelas de 24, 72, 168 e 336 horas (1, 3, 7 e 14 dias). A célula é fixada em tanh e sem dropout recorrente (a implementação rápida do cuDNN): a célula própria percorre cada passo em Python, e com janelas de centenas de horas e dezenas de milhares de amostras ficaria lenta demais para o tempo do Kaggle.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Eixo | Valores |
|---|---|
| `lookback` | `24`, `72`, `168`, `336` |

**4 combinações.** Fixos no bloco: `horizon=24`, `lstm_activation=tanh`, `recurrent_dropout=0`, `batch_size=256`, `epochs=100`, `patience=10`. Todas as configurações rodam os 5 folds.

**O que observar:** Se algum tamanho de janela bate o LSTM padrão por hora e a série diária (fase 3).

In [ ]:
if executar("fase8"):
    !python src/grid_search.py grids/f8_janela.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS $GRID_EXTRA

In [ ]:
rep.discarded_configs("f8_janela")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.diverged_configs("f8_janela")  # treinaram, mas divergiram (val/theil > limite): fora das médias

#### O que cada hiperparâmetro muda — `f8_janela`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("f8_janela")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("f8_janela")

#### Confirmação das finalistas e campeão — `f8_janela`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("f8_janela", "final")

In [ ]:
rep.show_champion("f8_janela")
rep.plot_finalists("f8_janela")

### 📝 Análise — Fase 8 · O melhor LSTM com dados por hora: tamanho da janela

- **Qual janela funcionou melhor?** _…_
- **Os dados por hora superaram a série diária?** _…_

In [ ]:
backup("f8_janela")

## Fase 8 · Teste revelado

Até aqui todas as escolhas desta fase usaram só a validação. Agora o período de teste é usado **uma única vez**, para as
referências e o campeão de cada bloco (média ± desvio entre os modelos dos K folds). A concordância entre validação e
teste é a evidência de que não houve vazamento; uma discrepância grande (ex.: mudança de regime) é um achado.

In [ ]:
final = rep.final_report(["f8_referencia", "f8_janela"])
final

In [ ]:
campeao_fase = rep.champion_name("f8_janela")
rep.plot_predictions(campeao_fase)

### O que descobrimos nesta fase

Na execução de referência:

- Ainda não executada.

**📝 Nesta execução:** _…_ (confira nas tabelas acima se os números se repetem)

# Comparação entre as fases

Para cada fase com treino: o passeio aleatório, o LSTM padrão e o campeão da fase, na validação e no teste. É o resumo de
tudo o que as tentativas de melhorar o modelo conseguiram.

In [ ]:
comparacao = rep.compare_phases()
comparacao

# Próximos passos

- Mudar o alvo: prever a volatilidade futura (sabidamente mais previsível que o retorno) ou a direção (classificação sobe/lateral/desce). Pede métricas próprias, porque Theil e POCID foram feitos para retorno.
- Dados sintéticos com TimeGAN, treinado dentro de cada fold só com dados de treino. Expectativa baixa: o gerador aprende com os mesmos dados, e o augmentation piorou 13 de 14 variações na fase 1; o que ajudou foi dado real a mais.

## 📝 Conclusões

- **Alguma tentativa fez o LSTM bater o passeio aleatório no teste de forma consistente? Qual e por quanto?** _…_
- **O que os hiperparâmetros da lista mudaram de fato (fase 1), e por que não se sustentou no teste?** _…_
- **Mais dados (fase 3) ou mais informação (fase 4): o que pesou mais?** _…_
- **Acerto de direção (POCID): em algum momento saiu de ~50%?** _…_